# DMV Lab: Code sheets → JSON → Excel table

This notebook reads every Word code sheet for one country from Dropbox. For each one it:

1. **transcribes** the code sheet into a JSON file (one per disaster, indented so a person can review it), using GPT;
2. **finds the disaster in the EM-DAT file** and writes its ID variables into the JSON (`EMDAT_Record`);
3. **builds the disaster-level Excel table** (Part B). EM-DAT sheet 1 is the baseline, and the code-sheet variables are added at the right edge, coded as in *Instructions for entering DMV data (AI version)*.

```
<DMV_ROOT>/<COUNTRY>/                         EM-DAT spreadsheet (the newest one is used)
<DMV_ROOT>/<COUNTRY>/Codesheets/              Word code sheets (.docx)         ← input
<DMV_ROOT>/<COUNTRY>/Data Files/JSONs/        codesheet_<DisNo>.json, _review_queue.xlsx   ← output
<DMV_ROOT>/<COUNTRY>/Data Files/              <COUNTRY>DisasterLevelofAnalysis<DDMMYYYY>.xlsx ← output
```

**Before the first run.** Add these Colab Secrets (key icon in the left sidebar): `OPENAI_API_KEY_DMV`, `DROPBOX_APP_KEY`, `DROPBOX_APP_SECRET` and `DROPBOX_REFRESH_TOKEN`. If you don't have a refresh token yet, the *Storage* cell walks you through getting one, once. The Dropbox app (dropbox.com/developers/apps) needs the permissions `files.metadata.read`, `files.content.read` and `files.content.write`. Then check the two *Settings* cells: `COUNTRY` and `DMV_ROOT`, the Dropbox folder that holds the country folders.

**What stays under human control**
* The JSON keeps the coder's answers exactly as typed. Nothing is "corrected" in it.
* Every JSON records whether it needs a human look (`Extraction_Info → needs_human_review`, `review_reasons`). All flagged files are listed in `Data Files/JSONs/_review_queue.xlsx`.
* After checking a JSON against its Word file, fix any wrong values in the JSON and set `"human_reviewed": true`. Re-running this notebook never overwrites a reviewed JSON.
* To change a value in the Excel table only (for example, giving a numeric code to an answer the rules cannot read), add it to the JSON's `Table_Overrides` block (`"reliefgoods": 2`), then rebuild the table. Don't edit the Excel file by hand: every rebuild starts again from the JSONs.
* The table workbook has a **Review** sheet (everything a person should check) and a **Coding_Audit** sheet (how each actor and state was counted).

Run the cells from top to bottom. The *Pilot run* cell does a few files first, so you can compare the JSONs against their Word files before the *Full run*. Both runs can be resumed: unchanged Word files that were already extracted are skipped.

### 1. Install & import
Installs the add-on packages the notebook needs (OpenAI, the Word reader, Dropbox, Excel tools) and loads them. Nothing is read or written yet. Run this first every time the notebook is opened.

In [ ]:
!pip install --upgrade --quiet openai python-docx dropbox pandas openpyxl pyxlsb xlrd tqdm

import os, re, io, json, time, random, hashlib, datetime, calendar, posixpath, unicodedata
from collections import Counter
from pathlib import Path
from zoneinfo import ZoneInfo

import pandas as pd
import dropbox
from tqdm.auto import tqdm

print("imports ready")

imports ready


### 2. Settings: country, folders, EM-DAT, table
The main settings: which country to process and where its folders are in Dropbox, which EM-DAT spreadsheet and sheet are the table's baseline, and whether code sheets for disasters found only on another sheet (of the EM-DAT Excel; for example epidemics on sheet 2) are left out of the table.  
*Note*: Change COUNTRY and DMV_ROOT here before running a new country.

In [ ]:
# ═════════════ COUNTRY AND WHERE THE FILES ARE ═════════════
COUNTRY  = "Bangladesh"          # the country folder's name (also EM-DAT's "Country" value)
STORAGE  = "dropbox"             # "dropbox" | "gdrive" | "local"
DMV_ROOT = "/DisastersProject"   # the folder that holds the country folders, written the way the backend expects:
                                 #   dropbox: path inside Dropbox starting with "/"  ("" = country folders at the top level)
                                 #   gdrive : "/content/drive/MyDrive/DMV Lab"      local: "C:/Users/you/DMV Lab"
DROPBOX_TEAM_SPACE = False  # True if the folders sit in a Dropbox team space / team folder

COUNTRY_FOLDER   = f"{DMV_ROOT.rstrip('/')}/{COUNTRY}"
CODESHEET_FOLDER = f"{COUNTRY_FOLDER}/Codesheets"     # Word code sheets
DATA_FOLDER      = f"{COUNTRY_FOLDER}/Data Files"     # the Excel table goes here (created if missing)
JSON_FOLDER      = f"{DATA_FOLDER}/JSONs"             # one JSON per code sheet (created if missing)

# ═════════════ EM-DAT ═════════════
EMDAT_FILE  = None   # None = the newest spreadsheet directly in COUNTRY_FOLDER with "EMDAT" in its name,
                     # or a file name / full path to use a specific one
EMDAT_SHEET = 0      # the table's baseline sheet: 0 = the first sheet ("import all the data from sheet 1").
                     # Other sheets are only searched when a code sheet's DisNo is not on that sheet.
ONLY_BASELINE_SHEET = True   # True  = only disasters on that sheet go in the table.
                             # A code sheet whose DisNo is on another sheet (e.g. epidemics or man-made disasters moved to sheet 2) is
                             #         left out of the table and listed on the Review sheet.
                             # False = such a disaster is added to the table as an extra row (flagged for review).
                             # Either way the other sheets are read, so their DisNos are recognised, not "unknown".

# ═════════════ EXCEL TABLE ═════════════
LAB_TIMEZONE = "America/Chicago"
TABLE_NAME   = None  # None = "<COUNTRY>DisasterLevelofAnalysis<DDMMYYYY>.xlsx" with today's date

print(f"country folder: {COUNTRY_FOLDER}  ({STORAGE})")

country folder: /DisastersProject/Bangladesh  (dropbox)


### 3. Settings: extraction (GPT)  
Settings for the GPT step only: the hand-checked example shown to GPT, the model name, how many times to retry, how many files the pilot run processes, and token prices for the cost estimate. None of these settings change how the Excel table is coded.

In [ ]:
RECURSIVE = False    # True = also read code sheets in subfolders of Codesheets/

# Few-shot example(s): hand-checked (Word code sheet, JSON) pairs on the same storage. [] = zero-shot.
# A missing example file is skipped with a warning.
FEWSHOT_PAIRS = [
    (f"{CODESHEET_FOLDER}/examples/Bangladesh1988JuneFloodCodesheet.docx",
     f"{CODESHEET_FOLDER}/examples/codesheet_1988-0242-BGD.txt"),
]

# Optional: codesheet_variable_dictionary.json written by dmv_reliefweb_pipeline.py (a local/Drive path),
# so both scripts use one variable list. None = use the list built into the template cell.
VARIABLE_DICTIONARY_JSON = None
# Optional: the ReliefWeb pipeline's outputs folder (local/Drive path). If set, the OCHA/ReliefWeb blocks of its
# codesheet_<DisNo>.txt are copied into the matching JSON (and fill the table's `ocharep`). None = they stay null.
RELIEFWEB_OUTPUTS_DIR = None

# ─────────── MODEL / API ───────────
MODEL             = "gpt-6-luna"
REASONING_EFFORT  = "low"      # this is transcription; try "medium" only if the pilot misplaces values
USE_STRICT_SCHEMA = True       # falls back to plain JSON mode automatically if the API rejects it
MAX_OUTPUT_TOKENS = 16000      # reasoning tokens share this budget
REQUEST_TIMEOUT   = 300
MAX_ATTEMPTS      = 3
PROMPT_VER        = "2026.10.08"

# ─────────── RUN BEHAVIOUR ───────────
PILOT_N        = 3       # the pilot cell runs this many files first
REDO_UNCHANGED = False   # False = skip Word files already extracted with identical content (resumable)
EMDAT_MISMATCH_REVIEW = 3   # flag a sheet when this many EM-DAT fields (other than the ones coders may revise)
                            # differ from what the coder typed: a sign of a wrong DisNo
PRICE = {"input": 2.00, "cached_input": 0.10, "output": 10.00}   # USD per 1M tokens

print(f"{CODESHEET_FOLDER} -> {JSON_FOLDER} | model={MODEL} ({REASONING_EFFORT})")

/DisastersProject/Bangladesh/Codesheets -> /DisastersProject/Bangladesh/Data Files/JSONs | model=gpt-6-luna (low)


### 4. Storage (Dropbox / Google Drive / local)
Connects to Dropbox (or Google Drive, or a folder on the computer) using the keys saved in Colab Secrets, so that the rest of the notebook reads and saves files the same way wherever they live. The cell (at the end) also checks that the country folder exists and is not empty.

In [ ]:
# One storage object with the same operations for each backend, so the rest of the code doesn't care where
# files live. Keys come from Colab Secrets (key icon in the sidebar).

# This function sets a key or password by name:
# first from the computer's environment, then from Colab Secrets;
# if it is still missing, asks you to type it.
def secret(name, ask=True):
    """Environment variable -> Colab Secret -> typed prompt (if ask)."""
    if os.environ.get(name):
        return os.environ[name]
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            return value
    except Exception:
        pass
    if ask:
        from getpass import getpass
        return getpass(f"{name}: ")
    return None

# Returns just the file name from a full path
def _name(path):
    return str(path).replace("\\", "/").rstrip("/").rsplit("/", 1)[-1]

# File operations (list, read, save, check, move) for a folder on this computer or on a mounted Google Drive.
class LocalStorage:
    """Local disk, or Google Drive once it is mounted in Colab (then Drive is just a folder)."""
    def list_entries(self, folder, recursive=False):
        root = Path(folder)
        if not root.is_dir():
            return []
        found = root.rglob("*") if recursive else root.glob("*")
        return sorted(({"path": str(p), "name": p.name, "is_dir": p.is_dir(),
                        "modified": datetime.datetime.fromtimestamp(p.stat().st_mtime, datetime.timezone.utc)
                                    .replace(tzinfo=None)} for p in found), key=lambda e: e["path"].lower())
    def list_files(self, folder, recursive=False):
        return [e["path"] for e in self.list_entries(folder, recursive) if not e["is_dir"]]
    def read_bytes(self, path):
        return Path(path).read_bytes()
    def write_bytes(self, path, data):
        p = Path(path)
        p.parent.mkdir(parents=True, exist_ok=True)
        tmp = p.with_name(p.name + ".tmp")
        tmp.write_bytes(data)
        os.replace(tmp, p)                      # atomic: a crash never leaves half a file
    def exists(self, path):
        return Path(path).exists()
    def move(self, src, dst):
        Path(dst).parent.mkdir(parents=True, exist_ok=True)
        os.replace(src, dst)
    def join(self, folder, name):
        return str(Path(folder) / name)

# The same file operations as LocalStorage, but on Dropbox, so the rest of the notebook works the
# same way whichever storage is used.
class DropboxStorage:
    """The same operations on Dropbox paths ("/DMV Lab/Bangladesh/file.docx")."""
    def __init__(self, dbx):
        self.dbx = dbx
    def list_entries(self, folder, recursive=False):
        folder = "" if folder in ("", "/") else folder.rstrip("/")
        try:
            res = self.dbx.files_list_folder(folder, recursive=recursive)
        except dropbox.exceptions.ApiError as e:
            if e.error.is_path() and e.error.get_path().is_not_found():
                return []
            raise
        entries = list(res.entries)
        while res.has_more:
            res = self.dbx.files_list_folder_continue(res.cursor)
            entries += res.entries
        out = []
        for e in entries:
            if isinstance(e, dropbox.files.FileMetadata):
                out.append({"path": e.path_display, "name": e.name, "is_dir": False, "modified": e.client_modified})
            elif isinstance(e, dropbox.files.FolderMetadata) and e.path_lower != folder.lower():
                out.append({"path": e.path_display, "name": e.name, "is_dir": True, "modified": None})
        return sorted(out, key=lambda x: x["path"].lower())
    def list_files(self, folder, recursive=False):
        return [e["path"] for e in self.list_entries(folder, recursive) if not e["is_dir"]]
    def read_bytes(self, path):
        return self.dbx.files_download(path)[1].content
    def write_bytes(self, path, data):
        self.dbx.files_upload(data, path, mode=dropbox.files.WriteMode.overwrite, mute=True)   # creates folders
    def exists(self, path):
        try:
            self.dbx.files_get_metadata(path)
            return True
        except dropbox.exceptions.ApiError as e:
            if e.error.is_path() and e.error.get_path().is_not_found():
                return False
            raise
    def move(self, src, dst):
        self.dbx.files_move_v2(src, dst, autorename=True)
    def join(self, folder, name):
        return posixpath.join(folder, name)

# One-time setup: walks the user through allowing this app to use the lab's Dropbox,
# then prints a long-lived token to save as the Colab Secret DROPBOX_REFRESH_TOKEN.
def dropbox_refresh_token_flow():
    """One time only: get a long-lived refresh token. Needs the DROPBOX_APP_KEY and DROPBOX_APP_SECRET secrets
    (Dropbox App Console -> your app -> Settings; on its Permissions tab tick files.metadata.read,
    files.content.read and files.content.write)."""
    print("No DROPBOX_REFRESH_TOKEN secret yet, so let's get one (only needed once).")
    flow = dropbox.DropboxOAuth2FlowNoRedirect(secret("DROPBOX_APP_KEY"), secret("DROPBOX_APP_SECRET"),
                                               token_access_type="offline")
    print("1) Open this link, sign in to the Dropbox account that has the DMV folders, click Allow:\n  ", flow.start())
    token = flow.finish(input("2) Paste the code Dropbox shows you: ").strip()).refresh_token
    print("\n3) Save this as the Colab Secret DROPBOX_REFRESH_TOKEN so you are not asked again:\n  ", token)
    return token

# Opens the storage chosen in the settings (Dropbox, Google Drive or a local folder) and returns the
# object the rest of the notebook uses to read and save files.
def connect_storage(kind):
    if kind == "dropbox":
        refresh, access = secret("DROPBOX_REFRESH_TOKEN", ask=False), secret("DROPBOX_ACCESS_TOKEN", ask=False)
        if not refresh and not access:
            refresh = dropbox_refresh_token_flow()
        if refresh:
            dbx = dropbox.Dropbox(oauth2_refresh_token=refresh,
                                  app_key=secret("DROPBOX_APP_KEY"), app_secret=secret("DROPBOX_APP_SECRET"))
        else:
            print("⚠️ using DROPBOX_ACCESS_TOKEN, which expires after ~4 hours")
            dbx = dropbox.Dropbox(oauth2_access_token=access)
        account = dbx.users_get_current_account()
        if DROPBOX_TEAM_SPACE:
            dbx = dbx.with_path_root(dropbox.common.PathRoot.root(account.root_info.root_namespace_id))
        print(f"Dropbox connected as {account.name.display_name}")
        return DropboxStorage(dbx)
    if kind == "gdrive":
        from google.colab import drive
        if not os.path.exists("/content/drive/MyDrive"):
            drive.mount("/content/drive")
        return LocalStorage()
    if kind == "local":
        return LocalStorage()
    raise ValueError(f"STORAGE must be 'dropbox', 'gdrive' or 'local', not {kind!r}")


storage = connect_storage(STORAGE)
_top = storage.list_entries(COUNTRY_FOLDER)
assert _top, (f"{COUNTRY_FOLDER} is empty or does not exist — check DMV_ROOT and COUNTRY in the settings cell "
              f"(top level has: {[e['name'] for e in storage.list_entries(DMV_ROOT or '/')][:30]})")
print(f"storage ready: {type(storage).__name__} | {COUNTRY_FOLDER} contains "
      f"{sum(e['is_dir'] for e in _top)} folder(s), {sum(not e['is_dir'] for e in _top)} file(s)")

No DROPBOX_REFRESH_TOKEN secret yet, so let's get one (only needed once).
1) Open this link, sign in to the Dropbox account that has the DMV folders, click Allow:
   https://www.dropbox.com/oauth2/authorize?response_type=code&client_id=b51qgad98fjaixj&token_access_type=offline
2) Paste the code Dropbox shows you: eTDrKCs6c2EAAAAAAAAAqnO87oDRjJpAgOjMzMc9lqI

[Dropbox token removed]
Dropbox connected as Eric Ngo
storage ready: DropboxStorage | /DisastersProject/Bangladesh contains 2 folder(s), 291 file(s)


### 5. EM-DAT file and shared value readers
Finds the newest EM-DAT spreadsheet in the country folder, reads every sheet that has a Dis. No. column, and keeps this country's rows from sheet 1, sorted by Dis No, as the table's baseline.  
The cell also defines small helpers used everywhere else: reading numbers such as "45,000,000", Yes/No answers and the dates coders type (e.g. "06/1988 – 09/1988"), and looking up a disaster by its Dis No.

In [ ]:
# ---------- small helpers shared by both parts ----------

# True if a value is empty: nothing, an empty Excel cell, or only spaces.
def is_blank(v):
    return v is None or (isinstance(v, float) and v != v) or (isinstance(v, str) and not v.strip())

# Makes text easy to compare: lower case, plain quotes and dashes, '45,000,000' -> '45000000', single spaces.
# Used only when comparing; stored text is never changed.
_FOLD = str.maketrans({"‘": "'", "’": "'", "“": '"', "”": '"',
                       "–": "-", "—": "-", "−": "-", " ": " "})
def fold(s):
    """For comparing text: lower case, plain quotes/dashes, no thousands commas, single spaces."""
    s = unicodedata.normalize("NFKC", str(s)).translate(_FOLD).lower()
    s = re.sub(r"(?<=\d),(?=\d{3}(?!\d))", "", s)            # 45,000,000 -> 45000000
    return re.sub(r"\s+", " ", s).strip()

# Turns a plain number such as 45000000, '45,000,000' or '2379' into a number.
# Anything else (e.g. 'approx. 2,000') gives None.
def to_number(v):
    """45000000 / '45,000,000' / '2379' -> int (or float); anything else -> None."""
    if isinstance(v, bool) or is_blank(v):
        return None
    if isinstance(v, (int, float)):
        return int(v) if float(v).is_integer() else v
    s = str(v).strip()
    if re.fullmatch(r"-?\d{1,3}(,\d{3})+(\.\d+)?|-?\d+(\.\d+)?", s):
        f = float(s.replace(",", ""))
        return int(f) if f.is_integer() else f
    return None

# Reads the Yes/No at the start of an answer: 'Yes , 09/01/1988' -> 'Yes', 'No (0)' -> 'No', otherwise None.
def yes_no_word(v):
    """'Yes , 09/01/1988' -> 'Yes';  'No (0)' -> 'No';  anything else -> None."""
    if is_blank(v):
        return None
    m = re.match(r"\s*(yes|y|no|n|none)\b", fold(v))
    return None if not m else ("Yes" if m.group(1) in ("yes", "y") else "No")

# ---------- dates written on code sheets: "06/1988 – 09/1988", "7/19/2019 – 7/22/2019", "June 3, 1988" ----------
_MON = (r"(?<![a-z])(?:jan(?:uary)?|feb(?:ruary)?|mar(?:ch)?|apr(?:il)?|may|june?|july?|aug(?:ust)?|"
        r"sep(?:t(?:ember)?)?|oct(?:ober)?|nov(?:ember)?|dec(?:ember)?)\.?")
_MONTHS = {**{m.lower(): i for i, m in enumerate(calendar.month_name) if m},
           **{m.lower(): i for i, m in enumerate(calendar.month_abbr) if m}, "sept": 9}
_SFX = r"(?:st|nd|rd|th)?"
_DATE_TOKEN = re.compile(rf"""
    (?P<iso>(?P<iy>\d{{4}})-(?P<im>\d{{1,2}})-(?P<id>\d{{1,2}}))
  | (?P<mdy>(?P<a>\d{{1,2}})[/.](?P<b>\d{{1,2}})[/.](?P<c>\d{{4}}))
  | (?P<my>(?P<mm>\d{{1,2}})/(?P<myy>\d{{4}}))
  | (?P<mddy>(?P<n1>{_MON})\s+(?P<d1>\d{{1,2}}){_SFX}\s*-\s*(?P<d2>\d{{1,2}}){_SFX},?\s+(?P<y1>\d{{4}}))
  | (?P<ddmy>(?P<d6>\d{{1,2}}){_SFX}\s*-\s*(?P<d7>\d{{1,2}}){_SFX}\s+(?P<n6>{_MON}),?\s+(?P<y6>\d{{4}}))
  | (?P<mdyn>(?P<n2>{_MON})\s+(?P<d3>\d{{1,2}}){_SFX},?\s+(?P<y2>\d{{4}}))
  | (?P<dmyn>(?P<d4>\d{{1,2}}){_SFX}\s+(?:of\s+)?(?P<n3>{_MON}),?\s+(?P<y3>\d{{4}}))
  | (?P<myn>(?P<n4>{_MON}),?\s+(?P<y4>\d{{4}}))
  | (?P<mdn>(?P<n5>{_MON})\s+(?P<d5>\d{{1,2}}){_SFX}(?![\d]))
  | (?P<mo>(?<![a-z])(?P<n7>january|february|march|april|june|july|august|september|october|november|december)(?![a-z]))
  | (?P<y>(?<!\d)(?:1[89]|20)\d{{2}}(?!\d))
""", re.I | re.X)

# Turns a month name or abbreviation ('June', 'Sept.') into its number (6, 9).
def _mon(name):
    n = name.lower().rstrip(".")
    return _MONTHS.get(n) or _MONTHS.get(n[:3])

# Reads the dates a coder typed ('06/1988 – 09/1988', 'June 3, 1988' ...) into start and end (year, month, day). Parts not given stay empty.
# If the text is unclear, or the end is before the start, it reports a problem instead of guessing.
def parse_dates(text):
    """Code-sheet date text -> {"start": (y, m, d), "end": (y, m, d) or None, "issue": str or None}.
    Missing parts are None. Month/Day/Year order is assumed (the template asks for it)."""
    out = {"start": None, "end": None, "issue": None}
    if is_blank(text) or fold(text) in ("--", "-", "n/a", "na", "unknown"):
        out["issue"] = "no date"
        return out
    s, toks, notes = fold(text), [], []
    for m in _DATE_TOKEN.finditer(s):
        g = m.groupdict()
        if g["iso"]:    toks.append((int(g["iy"]), int(g["im"]), int(g["id"])))
        elif g["mdy"]:
            a, b, y = int(g["a"]), int(g["b"]), int(g["c"])
            if a > 12 >= b:
                a, b = b, a
                notes.append(f"read {m.group(0)} as day/month/year")
            toks.append((y, a, b))
        elif g["my"]:   toks.append((int(g["myy"]), int(g["mm"]), None))
        elif g["mddy"]: toks += [(int(g["y1"]), _mon(g["n1"]), int(g["d1"])), (int(g["y1"]), _mon(g["n1"]), int(g["d2"]))]
        elif g["ddmy"]: toks += [(int(g["y6"]), _mon(g["n6"]), int(g["d6"])), (int(g["y6"]), _mon(g["n6"]), int(g["d7"]))]
        elif g["mdyn"]: toks.append((int(g["y2"]), _mon(g["n2"]), int(g["d3"])))
        elif g["dmyn"]: toks.append((int(g["y3"]), _mon(g["n3"]), int(g["d4"])))
        elif g["myn"]:  toks.append((int(g["y4"]), _mon(g["n4"]), None))
        elif g["mdn"]:  toks.append((None, _mon(g["n5"]), int(g["d5"])))
        elif g["mo"]:   toks.append((None, _mon(g["n7"]), None))      # "mid-June to September 1988"
        elif g["y"]:    toks.append((int(g["y"]), None, None))
    for i, (y, mo, d) in enumerate(toks):                   # "June 3 - June 10, 1988": borrow the year
        if y is None:
            y = next((t[0] for t in toks[i + 1:] if t[0]), None) or next((t[0] for t in toks[:i][::-1] if t[0]), None)
            toks[i] = (y, mo, d)
    bad = [t for t in toks if t[0] is None or (t[1] and not 1 <= t[1] <= 12) or (t[2] and not 1 <= t[2] <= 31)]
    if not toks or bad:
        out["issue"] = f"could not read a date in {text!r}"
        return out
    if len(toks) > 2:
        out["issue"] = f"more than two dates in {text!r}"
        return out
    out["start"] = toks[0]
    both = re.search(r"start\s*(&|and|/)\s*end|same day|one day|single day", s)
    out["end"] = toks[1] if len(toks) == 2 else (toks[0] if both else None)
    key = lambda t: (t[0], t[1] or 0, t[2] or 0)
    if out["end"] and key(out["end"]) < key(out["start"]):
        out["issue"] = f"end date before start date in {text!r}"
    if notes and not out["issue"]:
        out["note"] = "; ".join(notes)
    return out

# ---------- reading the EM-DAT spreadsheet ----------
SPREADSHEET_EXT = (".xlsx", ".xlsm", ".xls", ".xlsb", ".csv")

# Simplifies a column name for matching, e.g. "Total Damages ('000 US$)" -> 'total_damages_000_us'.
def norm_col(name):
    return re.sub(r"[^a-z0-9]+", "_", str(name).replace("’", "'").lower()).strip("_")

# field -> normalised EM-DAT column names (older exports like the Bangladesh file, and the current export)
EMDAT_COLS = {
    "disno": ["dis_no", "disno"], "year": ["year"], "seq": ["seq"], "glide": ["glide"],
    "country": ["country"], "iso": ["iso"], "region": ["region"], "continent": ["continent"],
    "subregion": ["subregion"],
    "group": ["disaster_group"], "subgroup": ["disaster_subgroup"], "type": ["disaster_type"],
    "subtype": ["disaster_subtype"], "subsubtype": ["disaster_subsubtype"], "event_name": ["event_name"],
    "location": ["location"], "origin": ["origin"], "river_basin": ["river_basin"],
    "ofda": ["ofda_bha_response", "ofda_response"], "appeal": ["appeal"], "declaration": ["declaration"],
    "aid": ["aid_contribution_000_us", "aid_contribution"], "lat": ["latitude"], "lon": ["longitude"],
    "sy": ["start_year"], "sm": ["start_month"], "sd": ["start_day"],
    "ey": ["end_year"], "em": ["end_month"], "ed": ["end_day"],
    "deaths": ["total_deaths"], "affected": ["total_affected"],
    "recon": ["reconstruction_costs_000_us"], "recon_adj": ["reconstruction_costs_adjusted_000_us"],
    "insured": ["insured_damage_000_us", "insured_damages_000_us"],
    "insured_adj": ["insured_damage_adjusted_000_us", "insured_damages_adjusted_000_us"],
    "damage": ["total_damage_000_us", "total_damages_000_us"],
    "damage_adj": ["total_damage_adjusted_000_us", "total_damages_adjusted_000_us"],
}
DATE_FIELDS = ["sy", "sm", "sd", "ey", "em", "ed"]

# Finds what this EM-DAT file calls a field (older and newer EM-DAT exports name some columns differently).
def emdat_col(columns, field):
    """The actual column name in this EM-DAT file for a field (None if the file has no such column)."""
    lookup = {norm_col(c): c for c in columns}
    return next((lookup[c] for c in EMDAT_COLS[field] if c in lookup), None)

# Reads one field from one EM-DAT row; None if the cell is empty.
def emdat_get(row, field):
    """Value of a field in an EM-DAT row (dict keyed by the file's own column names); None if blank."""
    col = emdat_col(row.keys(), field)
    v = row.get(col) if col else None
    return None if is_blank(v) else (v.strip() if isinstance(v, str) else v)

# Splits a Dis No into its number and country code:
# '1988-0242-BGD' -> ('1988-0242', 'BGD').
# A short number such as '1988-242' is padded to '1988-0242'.
def dis_key(s):
    """'1988-0242-BGD' -> ('1988-0242', 'BGD');  '1988-242' -> ('1988-0242', '')."""
    m = re.search(r"(\d{4})\s*-\s*(\d{1,4})(?:\s*-\s*([A-Za-z]{3}))?", str(s or ""))
    return (f"{m.group(1)}-{int(m.group(2)):04d}", (m.group(3) or "").upper()) if m else (None, "")

# Puts a Dis No back together: ('1988-0242', 'BGD') -> '1988-0242-BGD'.
def disno_text(event, iso):
    return f"{event}-{iso}" if iso else event

# Picks the EM-DAT spreadsheet: the file named in the settings, otherwise the newest spreadsheet in the
# country folder with 'EMDAT' in its name.
def find_emdat_file():
    if EMDAT_FILE:
        return EMDAT_FILE if "/" in str(EMDAT_FILE) or "\\" in str(EMDAT_FILE) else storage.join(COUNTRY_FOLDER, EMDAT_FILE)
    files = [e for e in storage.list_entries(COUNTRY_FOLDER) if not e["is_dir"]
             and e["name"].lower().endswith(SPREADSHEET_EXT) and not e["name"].startswith(("~$", "."))]
    named = [e for e in files if "emdat" in re.sub(r"[^a-z]", "", e["name"].lower())]
    cands = named or files
    assert cands, f"no EM-DAT spreadsheet directly in {COUNTRY_FOLDER} — set EMDAT_FILE in the settings cell"
    cands.sort(key=lambda e: e["modified"] or datetime.datetime.min, reverse=True)   # newest first
    if len(cands) > 1:
        print(f"  {len(cands)} EM-DAT spreadsheets found; using the newest. Others: "
              f"{[e['name'] for e in cands[1:]]} (set EMDAT_FILE to choose another)")
    return cands[0]["path"]

# Reads every sheet of the EM-DAT file that has a Dis No column, finding the header row even if there
# are title rows above it.
def read_emdat_book(path):
    """{sheet name: DataFrame} for every sheet with a DisNo column, keeping the file's own column names."""
    data, ext = storage.read_bytes(path), path.lower().rsplit(".", 1)[-1]
    if ext == "csv":
        frames = {"csv": pd.read_csv(io.BytesIO(data), dtype=object, header=None)}
    else:
        frames = pd.read_excel(io.BytesIO(data), sheet_name=None, dtype=object, header=None,
                               engine="pyxlsb" if ext == "xlsb" else None)
    book = {}
    for sheet, raw in frames.items():
        hdr = next((i for i in range(min(15, len(raw)))
                    if {norm_col(v) for v in raw.iloc[i] if pd.notna(v)} & {"dis_no", "disno"}), None)
        if hdr is None:
            print(f"  sheet {sheet!r}: no DisNo column, skipped")
            continue
        cols = [str(c).strip() if pd.notna(c) else f"Unnamed_{j}" for j, c in enumerate(raw.iloc[hdr])]
        df = raw.iloc[hdr + 1:].copy()
        df.columns = cols
        df = df.loc[:, [not (c.startswith("Unnamed_") and df[c].isna().all()) for c in df.columns]]
        df = df.loc[:, ~df.columns.duplicated()]
        disno_col = emdat_col(df.columns, "disno")
        df = df[df[disno_col].notna() & (df[disno_col].astype(str).str.strip() != "")]
        df = df.astype(object).where(df.notna(), None).reset_index(drop=True)
        book[sheet] = df
        print(f"  sheet {sheet!r}: {len(df)} disasters, {len(df.columns)} columns")
    return book

# Loads EM-DAT: sheet 1 (this country's rows only, sorted by Dis No) becomes the table's baseline, and the
# Dis Nos on every sheet go into a lookup list so code sheets can be matched to their disaster.
def load_emdat():
    """Reads the EM-DAT file into EMDAT_BASE (the table's baseline sheet) and EMDAT_INDEX (all sheets)."""
    global EMDAT_PATH, EMDAT_BOOK, EMDAT_BASE_SHEET, EMDAT_BASE, EMDAT_INDEX, EMDAT_BY_EVENT, HOME_ISO
    EMDAT_PATH = find_emdat_file()
    print(f"EM-DAT file: {_name(EMDAT_PATH)}")
    EMDAT_BOOK = read_emdat_book(EMDAT_PATH)
    assert EMDAT_BOOK, f"no sheet with a DisNo column in {_name(EMDAT_PATH)}"
    names = list(EMDAT_BOOK)
    EMDAT_BASE_SHEET = names[EMDAT_SHEET] if isinstance(EMDAT_SHEET, int) else EMDAT_SHEET
    base = EMDAT_BOOK[EMDAT_BASE_SHEET]
    ccol = emdat_col(base.columns, "country")
    if ccol:                                    # an export with several countries: keep this country only
        mine = base[base[ccol].astype(str).map(fold).str.startswith(fold(COUNTRY))]
        if 0 < len(mine) < len(base):
            print(f"  kept the {len(mine)} of {len(base)} rows whose Country is {COUNTRY}")
            base = mine
        elif len(mine) == 0:
            print(f"  ⚠️ no row has Country = {COUNTRY}; using all {len(base)} rows")
    dcol, icol = emdat_col(base.columns, "disno"), emdat_col(base.columns, "iso")
    isos = Counter(str(v).upper() for v in (base[icol] if icol else []) if not is_blank(v))
    isos = isos or Counter(dis_key(v)[1] for v in base[dcol] if dis_key(v)[1])
    HOME_ISO = isos.most_common(1)[0][0] if isos else ""
    EMDAT_BASE = base.sort_values(dcol, key=lambda s: s.astype(str)).reset_index(drop=True)
    EMDAT_INDEX, EMDAT_BY_EVENT = {}, {}
    for sheet in [EMDAT_BASE_SHEET] + [n for n in names if n != EMDAT_BASE_SHEET]:   # baseline sheet wins
        df = EMDAT_BASE if sheet == EMDAT_BASE_SHEET else EMDAT_BOOK[sheet]
        for row in df.to_dict("records"):
            event, iso = dis_key(emdat_get(row, "disno"))
            iso = iso or str(emdat_get(row, "iso") or "").upper()
            if event and (event, iso) not in EMDAT_INDEX:
                rec = {"sheet": sheet, "row": row}
                EMDAT_INDEX[(event, iso)] = rec
                EMDAT_BY_EVENT.setdefault(event, []).append(rec)
    print(f"baseline sheet {EMDAT_BASE_SHEET!r}: {len(EMDAT_BASE)} disasters | "
          f"{len(EMDAT_INDEX)} DisNos searchable across {len(names)} sheet(s)")

# Looks up a Dis No in EM-DAT and says how it matched (exact, by number only, or why it was not found).
# A YYYY-0000 number means a disaster the lab found that is not in EM-DAT.
def find_emdat(disno):
    """DisNo text -> (EM-DAT record {'sheet', 'row'} or None, how it matched)."""
    event, iso = dis_key(disno)
    if not event:
        return None, "no readable DisNo"
    if event.endswith("-0000"):
        return None, "not in EM-DAT (0000 = disaster found by the lab)"
    if iso and (event, iso) in EMDAT_INDEX:
        return EMDAT_INDEX[(event, iso)], "exact"
    recs = EMDAT_BY_EVENT.get(event, [])
    if len(recs) == 1:
        return recs[0], f"matched on {event} only (ISO on sheet: {iso or 'none'})"
    if len(recs) > 1:
        return None, f"{event} is in EM-DAT for {len(recs)} countries — add the ISO code"
    return None, f"{disno_text(event, iso)} is not in {_name(EMDAT_PATH)}"

load_emdat()

EM-DAT file: _BangladeshDisastersEMDAT2023.xlsb.xlsx
  sheet 'Sheet1': 329 disasters, 53 columns
  sheet 'Sheet2': 254 disasters, 52 columns
baseline sheet 'Sheet1': 329 disasters | 583 DisNos searchable across 2 sheet(s)


### 6. Output variable template & JSON schema
Lists the 46 code-sheet items (5 general-information items, then questions 1–41) in template order and gives each one a short key, e.g. question 20 becomes 20_gov_info_level.  
Also builds the fixed answer format GPT must follow, so every JSON has exactly the same fields in the same order.

In [ ]:
S_GI, S_DI = "General Information", "Disaster Information"
S_ACT, S_REG = "Disaster Response: Actors", "Disaster Response: Regulation"
S_COO, S_REL = "Disaster Response: Coordination", "Disaster Response: Relocation"
S_RES, S_REC = "Disaster Response: Restrictions", "Disaster Response: Reconstruction"

# (section, short_name, value_type, Word-template wording) — same order and names as CODESHEET_FIELDS in
# dmv_reliefweb_pipeline.py. Numbers are added below: General Information 01.., everything else 01..41
# (so "question 20" in the instructions is 20_gov_info_level).
FIELD_SPEC = [
    (S_GI, "country_name", "text", "Country Name"),
    (S_GI, "cow_country_code", "integer", "COW Country Code"),
    (S_GI, "emdat_disno", "text", "EM-DAT Disaster # (e.g., 1931-0010-HND)"),
    (S_GI, "non_emdat_source", "text", "Source if disaster identified from non-EM-DAT source"),
    (S_GI, "brief_description", "text", "Brief description of disaster"),
    (S_DI, "disaster_dates", "text", "Disaster Dates (Start Month/Day/Year & End Month/Day/Year)"),
    (S_DI, "disaster_group", "text", "Disaster Group"),
    (S_DI, "disaster_origin", "text", "Disaster Origin"),
    (S_DI, "disaster_subgroup", "text", "Disaster Subgroup"),
    (S_DI, "disaster_type", "text", "Disaster Type"),
    (S_DI, "disaster_subtype", "text", "Disaster Subtype"),
    (S_DI, "event_name", "text", "Event Name"),
    (S_DI, "lat_long", "text", "Disaster Latitude and Longitude"),
    (S_DI, "disaster_location", "text", "Disaster Location"),
    (S_DI, "total_deaths", "number", "Total Disaster Deaths"),
    (S_DI, "total_affected", "number", "Total Number of People Affected"),
    (S_DI, "aid_contribution_total_kusd", "number", "Aid Contribution Total"),
    (S_DI, "ofda_response", "text", "OFDA Response"),
    (S_DI, "appeal_intl_assistance", "text", "Appeal for International Assistance (Dates)"),
    (S_DI, "declaration_of_disaster", "text", "Declaration of Disaster (Dates)"),
    (S_DI, "river_basin", "text", "River Basin Name (if relevant)"),
    (S_DI, "reconstruction_costs_kusd", "number", "Reconstruction Costs"),
    (S_DI, "insured_damage_kusd", "number", "Insured Damage"),
    (S_DI, "total_damage_kusd", "number", "Total Damage"),
    (S_DI, "gov_info_level", "coded", "Level of government information about the disaster and damages (H/M/L/None)"),
    (S_DI, "shelters_provided", "coded", "Provision of shelters/approximate number of people sheltered (Y/N)"),
    (S_DI, "tourist_area", "coded", "Did the disaster occur in an area frequently visited by tourists? (Y/N)"),
    (S_DI, "relief_goods_level", "coded", "Other types of relief goods provided post-disaster (H/M/L/None)"),
    (S_ACT, "gov_actors", "list", "Government actors that responded to this disaster (e.g., city, regional, "
                                  "national government, ministries, military, leaders, central disaster admin)"),
    (S_ACT, "officials_visited", "list", "Leaders or other high-ranking officials that visited the area (list names)"),
    (S_ACT, "ngo_actors", "list", "NGO actors that responded to this disaster (domestic & international)"),
    (S_ACT, "io_actors", "list", "IO actors that responded to this disaster (regional & global)"),
    (S_ACT, "other_states", "list", "Other states that responded to this disaster"),
    (S_ACT, "aid_by_actor", "list", "Aid totals from each disaster actor"),
    (S_REG, "gov_requested_outside_aid", "coded", "Government requests for outside aid/assistance (Y/N)"),
    (S_REG, "gov_regulated_outside_aid", "coded", "Government regulation of outside aid or restrictions placed on outside funding (Y/N)"),
    (S_REG, "gov_response_failure", "coded", "Failure of government to respond to disaster (at any government level) (Y/N)"),
    (S_COO, "gov_outside_coordination_level", "coded", "Level of coordination between government and outside aid actors (H/M/L/None)"),
    (S_COO, "gov_coordination_of_aid_actors", "coded", "Government coordination of aid actors (H/M/L/None)"),
    (S_COO, "gov_led_meetings", "coded", "Government led meetings with aid actors (Y/N)"),
    (S_REL, "temporary_evacuations", "coded", "Temporary government evacuations of people in affected areas (Y/N)"),
    (S_REL, "permanent_relocation", "coded", "Permanent government relocation of people in affected areas (Y/N)"),
    (S_RES, "movement_restrictions", "coded", "Government restrictions on movements of people near disaster affected area (Y/N)"),
    (S_RES, "return_home_restrictions", "coded", "Government restrictions on return home for affected populations (Y/N)"),
    (S_REC, "reconstruction_resources_level", "coded", "Government resources committed to reconstruction of disaster area (H/M/L/None)"),
    (S_REC, "reconstruction_funding_level", "text", "Level of government funding for reconstruction money (e.g., national, regional, local)"),
]

# Numbers the items the way the code sheet does (General Information 01-05, then questions 01-41)
# and makes each key, e.g. '20_gov_info_level'.
def build_fields(spec):
    fields, gi, main = [], 0, 0
    for section, short, vtype, label in spec:
        if section == S_GI: gi += 1; n = gi
        else:               main += 1; n = main
        fields.append({"section": section, "key": f"{n:02d}_{short}", "value_type": vtype, "label": label})
    return fields

if VARIABLE_DICTIONARY_JSON and os.path.exists(VARIABLE_DICTIONARY_JSON):
    _d = json.load(open(VARIABLE_DICTIONARY_JSON, encoding="utf-8"))
    FIELDS = [{"section": r["section"], "key": r["key"], "value_type": r["value_type"], "label": r["original_label"]}
              for r in _d["fields"] if str(r.get("number", "")).strip()]
    print(f"variables loaded from {os.path.basename(VARIABLE_DICTIONARY_JSON)}")
else:
    FIELDS = build_fields(FIELD_SPEC)

KEYS = [f["key"] for f in FIELDS]
FIELD = {f["key"]: f for f in FIELDS}
assert len(set(KEYS)) == len(KEYS), "variable keys must be unique"
SHORT2KEY = {k.split("_", 1)[1]: k for k in KEYS}
SECTIONS = list(dict.fromkeys(f["section"] for f in FIELDS))
RELIEFWEB_BLOCKS = ["OCHA_API_Report_FileName", "OCHA_API_Report_FileLink", "ReliefWeb_Match_Info"]
# same root order as the example JSON, plus the EM-DAT record and the table overrides
ROOT_ORDER = [SECTIONS[0], *RELIEFWEB_BLOCKS[:2], *SECTIONS[1:], "Coder_Name", "Notes",
              "EMDAT_Record", "Table_Overrides", "ReliefWeb_Match_Info", "Extraction_Info"]

# ---- what the model returns: one FLAT object (Python nests it into sections afterwards) ----
_SCALAR = {"anyOf": [{"type": "string"}, {"type": "number"}, {"type": "null"}]}
_LIST   = {"type": "array", "items": {"type": "string"}}

# Helper for the answer format: an object that must contain exactly these fields and no others.
def _obj(props):
    return {"type": "object", "properties": props, "required": list(props), "additionalProperties": False}

MODEL_SCHEMA = _obj({
    **{k: (_LIST if FIELD[k]["value_type"] == "list" else _SCALAR) for k in KEYS},
    "Coder_Name":     {"anyOf": [{"type": "string"}, {"type": "null"}]},
    "Notes":          _LIST,
    "Unmapped_Items": {"type": "array", "items": _obj({"label": {"type": "string"}, "value": {"type": "string"}})},
    "Model_Comments": _LIST,
})

TEMPLATE_BLOCK = ""
for s in SECTIONS:
    TEMPLATE_BLOCK += f"\n[{s}]\n" + "".join(
        f"{f['key']} ({f['value_type']}): {f['label']}\n" for f in FIELDS if f["section"] == s)
TEMPLATE_BLOCK += "\nCoder_Name (text) | Notes (list) | Unmapped_Items (list of {label, value}) | Model_Comments (list)\n"

print(f"{len(FIELDS)} variables in {len(SECTIONS)} sections")

46 variables in 8 sections


### 7. Word reader (.docx → outline text)
Turns each Word code sheet into a plain-text outline GPT can read: italic section titles become headings and list items keep their indentation.  

Also reads tables and content boxes, includes tracked insertions and skips tracked deletions, so GPT sees the document as it currently reads in Word.

In [ ]:
# Turns each Word file into a clean outline: italic headings become "## …", list items become "- …" indented by
# their list level (the list numbers aren't needed: each item is matched by its question wording).
# Also reads tables, content controls, hyperlinks and tracked insertions, skips tracked deletions, and keeps
# curly quotes and dashes as the coder typed them.
from docx import Document
from docx.text.paragraph import Paragraph
from docx.oxml.ns import qn

_W = {n: qn(f"w:{n}") for n in ("p", "tbl", "tr", "tc", "sdt", "sdtContent", "r", "t", "tab", "br", "cr",
                                "noBreakHyphen", "rPr", "i", "rStyle", "val")}
_INVISIBLE = dict.fromkeys(map(ord, "​‌‍⁠﻿­"), None)

# Tidies text from Word: removes invisible characters and extra spaces, but keeps the coder's dashes and curly quotes.
def _clean(s):
    """Drop invisible characters, fold odd spaces/tabs to one space. Dashes and curly quotes are KEPT."""
    s = unicodedata.normalize("NFC", s).translate(_INVISIBLE)
    s = re.sub(r"[ \t    ]+", " ", s)
    return "\n".join(line.strip() for line in s.split("\n")).strip()

# Walks through the Word document in order, returning its paragraphs and tables, including those inside content boxes.
def _blocks(parent):
    """Paragraphs and tables in document order, including those inside content controls."""
    for el in parent.iterchildren():
        if el.tag in (_W["p"], _W["tbl"]):
            yield el
        elif el.tag == _W["sdt"]:
            content = el.find(_W["sdtContent"])
            if content is not None:
                yield from _blocks(content)

# Returns the pieces of text in one paragraph, including links and tracked insertions; tracked deletions are left out.
def _runs(p):
    """Runs of a paragraph, including runs inside hyperlinks and tracked insertions (deletions are skipped)."""
    for r in p.iter(_W["r"]):
        text = []
        for ch in r:
            if ch.tag == _W["t"]:                    text.append(ch.text or "")
            elif ch.tag == _W["tab"]:                text.append("\t")
            elif ch.tag in (_W["br"], _W["cr"]):     text.append("\n")
            elif ch.tag == _W["noBreakHyphen"]:      text.append("-")
        yield r, "".join(text)

# Checks whether a Word style, or the style it is based on, is set to italic.
def _style_italic(style):
    while style is not None:
        if getattr(style, "font", None) is not None and style.font.italic is not None:
            return style.font.italic
        style = getattr(style, "base_style", None)
    return None

# Checks whether a piece of text is italic, either set directly or through its style.
def _run_italic(r, para_style, char_styles):
    rPr = r.find(_W["rPr"])
    if rPr is not None:
        i = rPr.find(_W["i"])
        if i is not None:
            return i.get(_W["val"]) not in ("0", "false", "off")
        rs = rPr.find(_W["rStyle"])
        if rs is not None:
            v = _style_italic(char_styles.get(rs.get(_W["val"])))
            if v is not None:
                return v
    return bool(_style_italic(para_style))

# How deep a list item is (0 = top level, 1 = sub-item, ...); None if the paragraph is not a list item.
def _list_level(p, para_style):
    """0, 1, 2 ... for Word list items (numbered or bulleted); None for ordinary paragraphs."""
    numPr = p.pPr.numPr if p.pPr is not None else None
    style = para_style
    while numPr is None and style is not None:
        spPr = style.element.pPr
        numPr = spPr.numPr if spPr is not None else None
        style = style.base_style
    if numPr is None or numPr.numId is None or numPr.numId.val == 0:
        return None
    return numPr.ilvl.val if numPr.ilvl is not None else 0

# Reads one Word code sheet and labels every line as a heading, list item, table row or plain paragraph.
def read_codesheet(data: bytes):
    """Word bytes -> (outline text for the model, list of line dicts for the review checks)."""
    doc = Document(io.BytesIO(data))
    char_styles = {s.style_id: s for s in doc.styles if hasattr(s, "font")}
    lines = []
    for el in _blocks(doc.element.body):
        if el.tag == _W["tbl"]:
            for tr in el.iter(_W["tr"]):
                cells = [_clean(" ".join("".join(t for _, t in _runs(p)) for p in tc.iter(_W["p"])))
                         for tc in tr.iterchildren(_W["tc"])]
                cells = [c for i, c in enumerate(cells) if c and (i == 0 or c != cells[i - 1])]
                if cells:
                    lines.append({"kind": "table", "level": 0, "text": " | ".join(cells), "cells": cells})
            continue
        para = Paragraph(el, doc)
        style = para.style
        runs = [(r, t) for r, t in _runs(el)]
        text = _clean("".join(t for _, t in runs))
        if not text:
            continue
        level = _list_level(el, style)
        if level is not None:
            lines.append({"kind": "item", "level": level, "text": text})
            continue
        style_name = (style.name or "").lower() if style is not None else ""
        italic = all(_run_italic(r, style, char_styles) for r, t in runs if t.strip())
        heading = len(text) < 150 and (italic or style_name.startswith(("heading", "title")))
        lines.append({"kind": "heading" if heading else "para", "level": 0, "text": text})
    return render_outline(lines), lines

# Writes the labelled lines as an outline: '## ' for headings, '- ' indented by level for list items,
# '| ' for table rows.
def render_outline(lines):
    out = []
    for ln in lines:
        if ln["kind"] == "heading":
            out.append(f"\n## {ln['text']}")
        elif ln["kind"] == "item":
            ind = "  " * ln["level"]
            out.append(f"{ind}- " + ln["text"].replace("\n", "\n" + ind + "  "))
        elif ln["kind"] == "table":
            out.append(f"| {ln['text']} |")
        else:
            out.append(ln["text"])
    return "\n".join(out).strip()

print("Word reader ready")

Word reader ready


### 8. List the code sheets and preview one
Lists every .docx file in the Codesheets folder and warns about files it will skip (old .doc files, PDFs, subfolders). Then prints the first code sheet as GPT will see it, so users can check the Word reader worked before any money is spent on GPT.

In [ ]:
_entries = storage.list_entries(CODESHEET_FOLDER, RECURSIVE)
_all = [e["path"] for e in _entries if not e["is_dir"]]
DOCS = [p for p in _all if p.lower().endswith(".docx") and not _name(p).startswith(("~$", "."))]
_other = [p for p in _all if p.lower().endswith((".doc", ".gdoc", ".docm", ".odt", ".rtf", ".pdf"))]
_subfolders = [e["name"] for e in _entries if e["is_dir"]]

print(f"{len(DOCS)} .docx code sheet(s) in {CODESHEET_FOLDER}")
if _other:
    print(f"⚠️ {len(_other)} file(s) skipped — save them as .docx to include them:")
    for p in _other[:20]: print("   ", _name(p))
if _subfolders and not RECURSIVE:
    print(f"ℹ️ {len(_subfolders)} subfolder(s) not read (set RECURSIVE = True to include them): {_subfolders[:10]}")
assert DOCS, f"no .docx files in {CODESHEET_FOLDER} — check the folder name in the settings cell"

PREVIEW = DOCS[0]
outline, _lines = read_codesheet(storage.read_bytes(PREVIEW))
print(f"\n── outline of {_name(PREVIEW)} ({len(outline):,} chars, {len(_lines)} lines) ──\n")
print(outline[:5000])

147 .docx code sheet(s) in /DisastersProject/Bangladesh/Codesheets
ℹ️ 1 subfolder(s) not read (set RECURSIVE = True to include them): ['examples']

── outline of Bangladesh1904NovemberCycloneCodesheet.docx (3,461 chars, 62 lines) ──

DMV Lab Disaster Level Information Sheet
Note: underlined variables can be found in EM-DAT database.

## General Information
- Country Name : Bangladesh
- COW Country Code: 771
- EM-DAT Disaster # (e.g., 1931-0010-HND): 1904-0003-BGD
- Source if disaster identified from non-EM-DAT source: --
- Brief description of disaster: This tropical cyclone destroyed a fishing fleet and killed 143 people. It primarily affected the area of Sonadia in Chittagong, a district in Southern Bangladesh.
  - Description derived from source listed in notes below.

## Disaster Information (supplement with external information where information is missing)
- Disaster Dates (Start Month/Day/Year & End Month/Day/Year): 11/1904 – 11/1904
  - No further details about exact date in EM

### 9. LLM prompt & few-shot examples
Writes the instructions GPT receives: copy the coder's answers exactly, never correct, summarise or add anything, and put content that fits no question under Unmapped_Items.  
Also adds the hand-checked example (the June 1988 flood code sheet and its JSON) so GPT sees one correct answer as example before doing the rest.

In [ ]:
SYSTEM_PROMPT = """You transcribe DMV Lab disaster code sheets into a fixed JSON template. You are a transcriber, not a coder: never infer, correct, summarize, or add information.

INPUT: a code sheet converted from Word. Lines starting with "## " are section headings (italic in Word). Lines starting with "- " are list items; each extra 2 spaces of indentation is one level deeper. A top-level item normally reads "Question wording: coder's answer".

RULES
1. Match each question to the template key whose wording it matches. Wording may differ slightly between template versions; match on meaning, never on position alone.
2. Copy answers exactly: keep spelling mistakes, capitalization, punctuation, dashes, quotes, currency, and codes such as "Medium (2)" or "Yes (1)". Only trim spaces at the ends.
3. Types "number" and "integer": if the answer is a plain number, return a JSON number without thousands separators ("45,000,000" -> 45000000). Otherwise copy the text ("--", "approx. 2,000").
4. An empty answer (nothing after the colon and no sub-items), or a question missing from the sheet -> null (for list keys: []). "--" is an answer: copy it as "--".
5. Type "list": each sub-item under the question is one string, in document order; never merge, reorder, or deduplicate items. If a sub-item has its own deeper sub-items, append them to it separated by "; ". If the coder wrote the list on one line, split it only where they clearly separated entries (semicolons, or commas between names). The lab counts these entries (government actors, leaders, NGOs, IOs, other states), so one entry = one actor as the coder wrote it: never split inside a name or its details ("Ministry of Relief and Rehabilitation – Minister Kazi Feroz Rashif", "Bangladesh Red Cross Society/Red Cresent Society" and "Local authorities (ex. The Cuzco department)" are one entry each).
6. Any question: if the answer after the colon is empty but the question has sub-items, the sub-items ARE the answer (for text/number/coded keys join them with a line break). If the question already has an answer after the colon, its sub-items are explanations: put each in Notes as "[<key>] <text>", e.g. "[32_gov_response_failure] Government ...".
7. Notes: first the items of the final "Notes" section in order (except the "Coded by ..." line), then the "[<key>]" notes from rule 6 in document order.
8. Coder_Name: the name(s) on the "Coded by ..." line without the words "Coded by"; null if there is none.
9. Unmapped_Items: every question or line with coder content that fits no key (e.g., a question added in a newer template), as {"label", "value"} copied exactly. Never force such content into a key. Ignore the sheet title and template instructions such as "Note: underlined variables can be found in EM-DAT database."
10. Model_Comments: short notes on anything a human reviewer should check (an answer that could belong to two keys, a question that appears twice, an answer that looks misplaced). [] if none.

Return only the JSON object with every key of the template."""

def to_model_format(gold: dict) -> dict:
    """A finished code-sheet JSON (like the example) -> the flat object the model is asked to return."""
    flat = {}
    for f in FIELDS:
        v = (gold.get(f["section"]) or {}).get(f["key"])
        flat[f["key"]] = (v if isinstance(v, list) else []) if f["value_type"] == "list" else v
    flat["Coder_Name"] = gold.get("Coder_Name") or None
    flat["Notes"] = gold.get("Notes") or []
    flat["Unmapped_Items"] = (gold.get("Extraction_Info") or {}).get("unmapped_items") or []
    flat["Model_Comments"] = []
    return flat

FEWSHOT_HASHES = set()          # a few-shot sheet that is also in the input folder gets flagged in its output
EXAMPLES_BLOCK, _n_examples = "", 0
for docx_path, json_path in FEWSHOT_PAIRS:
    if not (storage.exists(docx_path) and storage.exists(json_path)):
        print(f"⚠️ few-shot example not found, skipped: {_name(docx_path)} / {_name(json_path)}")
        continue
    _n_examples += 1
    raw = storage.read_bytes(docx_path)
    FEWSHOT_HASHES.add(hashlib.sha256(raw).hexdigest())
    ex_outline, _ = read_codesheet(raw)
    ex_json = to_model_format(json.loads(storage.read_bytes(json_path).decode("utf-8-sig")))
    EXAMPLES_BLOCK += (f"\n# Example {_n_examples} input\n{ex_outline}\n\n# Example {_n_examples} output\n"
                       f"{json.dumps(ex_json, ensure_ascii=False, indent=1)}\n")

# Everything before the code sheet is identical on every request -> this is the part the prompt cache reuses.
STATIC_PREFIX    = f"# Template keys (key (type): Word-template wording)\n{TEMPLATE_BLOCK}{EXAMPLES_BLOCK}"
PROMPT_HASH      = hashlib.sha256((SYSTEM_PROMPT + STATIC_PREFIX).encode("utf-8")).hexdigest()[:12]
PROMPT_CACHE_KEY = f"dmv-codesheet-{PROMPT_HASH}"
print(f"few-shot examples: {_n_examples} | static prefix: {len(STATIC_PREFIX):,} chars | prompt hash {PROMPT_HASH}")

few-shot examples: 1 | static prefix: 15,477 chars | prompt hash 73b412971b3b


### 10. OpenAI client & API caller
Connects to OpenAI with the lab's API key and defines how one code sheet is sent and the answer received. If the API rejects an optional setting, the notebook switches that setting off and tries again; after 3 failed attempts the cell moves on to the next file instead of stopping.

In [ ]:
from openai import OpenAI
client = OpenAI(api_key=secret("OPENAI_API_KEY_DMV"), timeout=REQUEST_TIMEOUT)

UNSUPPORTED = set()            # features the API rejected; switched off for the rest of the run
STRICT_ON   = USE_STRICT_SCHEMA

# Assembles one request to OpenAI: the instructions, template and example (identical for every file, so
# OpenAI can cache them and bill them cheaply), then this code sheet's text and the required answer format.
def build_body(outline: str) -> dict:
    explicit_cache = not ({"prompt_cache_breakpoint", "prompt_cache_options"} & UNSUPPORTED)
    prefix = {"type": "input_text", "text": STATIC_PREFIX}
    if explicit_cache:
        prefix["prompt_cache_breakpoint"] = {"mode": "explicit"}
    body = {
        "model": MODEL,
        "input": [{"role": "system", "content": SYSTEM_PROMPT},
                  {"role": "user", "content": [prefix, {"type": "input_text",
                                                        "text": f"# Code sheet to transcribe\n{outline}"}]}],
        "max_output_tokens": MAX_OUTPUT_TOKENS,
        "text": {"format": {"type": "json_schema", "name": "dmv_codesheet", "schema": MODEL_SCHEMA, "strict": True}
                 if STRICT_ON else {"type": "json_object"}},
    }
    if explicit_cache:
        body["prompt_cache_options"] = {"mode": "explicit", "ttl": "30m"}
    if "prompt_cache_key" not in UNSUPPORTED:
        body["prompt_cache_key"] = PROMPT_CACHE_KEY
    if "reasoning" not in UNSUPPORTED:
        body["reasoning"] = {"effort": REASONING_EFFORT}
    return body

_SIGNALS   = ("unsupported", "unknown parameter", "unrecognized", "not supported", "unexpected keyword",
              "invalid schema")
_DROPPABLE = ("prompt_cache_breakpoint", "prompt_cache_options", "prompt_cache_key", "reasoning")

# If OpenAI rejects an optional setting (strict answer format, caching, reasoning effort), switches it off for
# the rest of the run so the same request can be retried.
def _adapt(msg: str) -> bool:
    """If the error says a feature was rejected, switch it off and return True (retry without using an attempt)."""
    global STRICT_ON
    low = msg.lower()
    if not any(s in low for s in _SIGNALS):
        return False
    if STRICT_ON and ("schema" in low or "text.format" in low):
        STRICT_ON = False
        print("  [adapt] strict schema rejected -> plain JSON mode ⚠️ (the post-processing still enforces the template)")
        return True
    for p in _DROPPABLE:
        if p not in UNSUPPORTED and re.search(rf"(?<![a-z_]){re.escape(p)}(?![a-z_])", low):
            UNSUPPORTED.add(p)
            print(f"  [adapt] dropping unsupported parameter: {p}")
            return True
    return False

# Pulls the answer text, any refusal, and the token counts out of OpenAI's reply.
def parse_response(d: dict) -> dict:
    out = {"status": "ok", "text": "", "refusal": None, "response_id": d.get("id"),
           "finish": (d.get("incomplete_details") or {}).get("reason") or d.get("status")}
    for blk in d.get("output") or []:
        if blk.get("type") == "message":
            for c in blk.get("content") or []:
                if c.get("type") == "output_text": out["text"] += c.get("text") or ""
                elif c.get("type") == "refusal":   out["refusal"] = c.get("refusal")
    u = d.get("usage") or {}
    out["usage"] = {"input_tokens": u.get("input_tokens"),
                    "cached_tokens": (u.get("input_tokens_details") or {}).get("cached_tokens"),
                    "output_tokens": u.get("output_tokens"),
                    "reasoning_tokens": (u.get("output_tokens_details") or {}).get("reasoning_tokens")}
    return out

# Sends one code sheet to GPT, retrying up to MAX_ATTEMPTS times with a short pause between tries.
# Never stops the run: if every try fails it returns an error record instead.
def call_model(outline: str) -> dict:
    """Never raises. Returns parse_response(...) or {'status': 'api_error', ...}."""
    attempt, last = 0, None
    while attempt < MAX_ATTEMPTS:
        try:
            return parse_response(client.responses.create(**build_body(outline)).model_dump())
        except Exception as e:
            if _adapt(str(e)):
                continue
            last, attempt = f"{type(e).__name__}: {str(e)[:500]}", attempt + 1
            if attempt < MAX_ATTEMPTS:
                time.sleep(min(2 ** attempt, 30) + random.uniform(0, 1))
    return {"status": "api_error", "error": last, "text": "", "usage": {}}

print("OpenAI client ready")

OpenAI client ready


### 11. Post-processing, review checks & EM-DAT record
Tidies GPT's answer into the template (every field present, numbers as numbers) without changing the coder's wording.  
Then runs automatic checks that flag a file for human review: text in the JSON that is not in the Word file, Word text missing from the JSON, odd values, and signs of a wrong Dis No (file name, dates, type or country that don't match EM-DAT).  
Finally, the cell finds the disaster's EM-DAT row and records where the coder's values differ from it.

In [ ]:
# ---------- parse + coerce the model's answer onto the template ----------
# Removes ``` code-block markers if GPT wrapped its answer in them.
def strip_fences(t):
    t = (t or "").strip()
    if t.startswith("```"):
        t = re.sub(r"^```[a-zA-Z]*\s*", "", t)
        t = re.sub(r"\s*```$", "", t)
    return t.strip()

# Reads GPT's answer as JSON; if there is extra text around it, tries the part from the first '{' to the last '}'.
def parse_json(text):
    s = strip_fences(text)
    try:
        return json.loads(s)
    except Exception:
        m = re.search(r"\{.*\}", s, re.DOTALL)
        try:
            return json.loads(m.group(0)) if m else None
        except Exception:
            return None

# Turns '45,000,000' into 45000000; anything that is not a plain number is left exactly as typed.
def _num(v):
    """'45,000,000' -> 45000000; anything that is not a plain number is returned unchanged."""
    n = to_number(v) if isinstance(v, (str, float)) else None
    return v if n is None else n

# Fits GPT's answer onto the template: every question present, lists as lists, numbers as numbers.
# The coder's wording is not changed; every adjustment is noted for the review list.
def coerce(obj):
    """Model output -> flat dict with every key and the right type. Returns (flat, notes)."""
    notes, flat = [], {}
    for k in KEYS:
        v, vtype = obj.get(k), FIELD[k]["value_type"]
        if k not in obj:
            notes.append(f"{k}: missing in model output")
        if vtype == "list":
            if isinstance(v, str):
                notes.append(f"{k}: string -> list"); v = [v]
            flat[k] = [str(x).strip() for x in (v or []) if x is not None and str(x).strip()]
        else:
            if isinstance(v, list):
                notes.append(f"{k}: list -> string"); v = "\n".join(str(x) for x in v)
            if isinstance(v, str):
                v = v.strip() or None
            flat[k] = _num(v) if vtype in ("number", "integer") else v
    flat["Coder_Name"] = (str(obj.get("Coder_Name")).strip() or None) if obj.get("Coder_Name") else None
    flat["Notes"] = [str(x).strip() for x in (obj.get("Notes") or []) if str(x).strip()]
    flat["Unmapped_Items"] = [{"label": str(u.get("label", "")).strip(), "value": str(u.get("value", "")).strip()}
                              for u in (obj.get("Unmapped_Items") or []) if isinstance(u, dict)]
    flat["Model_Comments"] = [str(x).strip() for x in (obj.get("Model_Comments") or []) if str(x).strip()]
    extra = [k for k in obj if k not in flat]
    if extra:
        notes.append(f"dropped unknown keys: {extra}")
    return flat, notes

# ---------- checks for human review (text matching only, never changes a value) ----------

# Splits a value at line breaks and semicolons, so each piece can be checked against the Word text.
def _pieces(v):
    return [p.strip() for p in re.split(r"[\n;]", str(v)) if len(p.strip()) >= 3]

# Lists every value in the extracted JSON (answers, notes, coder name, unmapped items) for the review checks.
def _output_values(flat):
    for k in KEYS:
        v = flat[k]
        for x in (v if isinstance(v, list) else [v]):
            if x is not None and str(x).strip() != "--":
                yield k, str(x)
    for x in flat["Notes"]:
        yield "Notes", re.sub(r"^\[\d{2}_[a-z_]+\]\s*", "", x)
    if flat["Coder_Name"]:
        yield "Coder_Name", flat["Coder_Name"]
    for u in flat["Unmapped_Items"]:
        yield "Unmapped_Items", u["value"]

BOILERPLATE = [r"^dmv lab disaster level information sheet", r"^note: underlined variables", r"^notes?:?$"]
_CODED_OK = re.compile(r"^\s*(--|yes|no|high|medium|low|none|[hml]\b|[0-3]\b)", re.I)

# Automatic checks of GPT's output against the Word text: values not found in the Word file, Word text
# missing from the JSON, a malformed Dis No, text in number fields, unexpected coded answers, and nearly
# empty sheets. It only flags problems; it never changes a value.
def review_checks(flat, lines, outline):
    """flat = what the model read from the Word sheet, checked against the Word text."""
    reasons = []
    src = fold(outline)
    # 1) every extracted value must appear in the Word text (catches "corrected" spelling or invented text)
    not_verbatim = sorted({k for k, v in _output_values(flat) if any(fold(p) not in src for p in _pieces(v))})
    if not_verbatim:
        reasons.append(f"not verbatim in source: {not_verbatim}")
    # 2) every answer in the Word text must appear somewhere in the output (catches dropped content)
    outs = [fold(v) for _, v in _output_values(flat)]
    out_pieces = [fold(p) for _, v in _output_values(flat) for p in _pieces(v)]
    blob = " ␞ ".join(outs)
    dropped = []
    for ln in lines:
        if ln["kind"] == "heading" or any(re.search(b, fold(ln["text"])) for b in BOILERPLATE):
            continue
        if ln["kind"] == "table":
            texts = ln["cells"][1:] or ln["cells"]
        elif ln["level"] == 0 and ":" in ln["text"] and len(ln["text"].split(":", 1)[0]) <= 200:
            texts = [ln["text"].split(":", 1)[1]]          # "Question: answer" -> answer only
        else:
            texts = [ln["text"]]
        for t in texts:
            for p in _pieces(t):
                f = fold(p)
                if len(f) < 4 or f in ("--", "n/a") or f in blob:
                    continue
                if any(len(o) >= 0.5 * len(f) and o in f for o in out_pieces):
                    continue
                dropped.append(p[:80])
    if dropped:
        reasons.append(f"source text not in output ({len(dropped)}): {dropped[:5]}")
    # 3) value-format checks
    disno = str(flat.get(SHORT2KEY["emdat_disno"]) or "")
    if not re.fullmatch(r"\d{4}-\d{4}(-[A-Z]{3})?", disno):
        reasons.append(f"EM-DAT DisNo missing or malformed on the sheet: {disno!r}")
    for k in KEYS:
        v, vtype = flat[k], FIELD[k]["value_type"]
        if vtype in ("number", "integer") and isinstance(v, str) and v != "--":
            reasons.append(f"non-numeric {k}: {v[:40]!r}")
        if vtype == "coded" and isinstance(v, str) and not _CODED_OK.match(v):
            reasons.append(f"unexpected coded value {k}: {v[:40]!r}")
    coder_keys = [k for k in KEYS if FIELD[k]["value_type"] in ("coded", "list")]     # what only the coder fills in
    filled = sum(1 for k in coder_keys if flat[k] not in (None, [], ""))
    if filled < 0.3 * len(coder_keys):
        reasons.append(f"mostly empty ({filled}/{len(coder_keys)} coded/list questions answered): blank template?")
    if flat["Unmapped_Items"]:
        reasons.append(f"{len(flat['Unmapped_Items'])} unmapped item(s): "
                       f"{[u['label'][:40] for u in flat['Unmapped_Items']][:5]}")
    if flat["Model_Comments"]:
        reasons.append(f"model comments: {flat['Model_Comments'][:3]}")
    return reasons

# ---------- optional: OCHA/ReliefWeb blocks from the ReliefWeb pipeline's codesheet_<DisNo>.txt ----------

# Optional: copies the OCHA/ReliefWeb report details found by the ReliefWeb pipeline into this disaster's JSON.
_RW_INDEX = None
def reliefweb_blocks(disno):
    global _RW_INDEX
    if not RELIEFWEB_OUTPUTS_DIR or not disno:
        return {}
    if _RW_INDEX is None:
        _RW_INDEX = {p.stem[len("codesheet_"):]: p for p in Path(RELIEFWEB_OUTPUTS_DIR).rglob("codesheet_*.txt")}
    p = _RW_INDEX.get(disno)
    return {k: json.loads(p.read_text(encoding="utf-8")).get(k) for k in RELIEFWEB_BLOCKS} if p else {}

# ---------- EM-DAT record: which EM-DAT row the sheet belongs to, and where the coder differs ----------
# The JSON keeps the coder's values exactly as typed. The table step decides which value goes in the table.
CODER_MAY_REVISE = {"disaster_dates", "total_deaths", "total_affected", "declaration_of_disaster"}
EMDAT_SNAPSHOT = ["disno", "year", "seq", "glide", "country", "iso", "group", "subgroup", "type", "subtype",
                  "subsubtype", "event_name", "location", "origin", "sy", "sm", "sd", "ey", "em", "ed",
                  "deaths", "affected", "ofda", "appeal", "declaration"]
_SIMPLE = {"disaster_group": "group", "disaster_origin": "origin", "disaster_subgroup": "subgroup",
           "disaster_type": "type", "event_name": "event_name", "disaster_location": "location",
           "total_deaths": "deaths", "total_affected": "affected", "aid_contribution_total_kusd": "aid",
           "river_basin": "river_basin"}
_YESNO = {"ofda_response": "ofda", "appeal_intl_assistance": "appeal", "declaration_of_disaster": "declaration"}
_MONEY = {"reconstruction_costs_kusd": ("recon", "recon_adj"), "insured_damage_kusd": ("insured", "insured_adj"),
          "total_damage_kusd": ("damage", "damage_adj")}
_FUZZY = {"disaster_location", "event_name", "disaster_origin"}      # wording often differs; not a wrong-DisNo sign

# True if two values match: equal numbers, or the same text ignoring case, quotes and spacing.
def _same(a, b):
    na, nb = to_number(a), to_number(b)
    if na is not None and nb is not None:
        return float(na) == float(nb)
    return b is not None and fold(a) == fold(b)

# Writes an EM-DAT row's start and end dates as text (month/day/year) for the review notes.
def _date_text(row):
    def one(y, m, d):
        return "/".join(str(x) for x in (m, d, y) if x is not None) if y else None
    g = [to_number(emdat_get(row, f)) for f in DATE_FIELDS]
    return f"{one(*g[:3]) or '?'} – {one(*g[3:]) or '?'}"

# True if the coder's dates agree with EM-DAT (parts the coder left out are not compared).
def dates_agree(text, row):
    p = parse_dates(text)
    if p["issue"]:
        return False
    em = [to_number(emdat_get(row, f)) for f in DATE_FIELDS]
    pairs = list(zip(p["start"], em[:3])) + (list(zip(p["end"], em[3:])) if p["end"] else [])
    return all(c is None or c == e for c, e in pairs)

# Compares the coder's copies of the EM-DAT fields (underlined on the code sheet) with the EM-DAT row and
# lists every difference. Nothing is changed here; the table step decides which value is used.
def compare_with_emdat(flat, row):
    diffs = []
    for short in list(_SIMPLE) + list(_YESNO) + list(_MONEY) + ["disaster_dates", "disaster_subtype", "lat_long"]:
        cs = flat.get(SHORT2KEY[short])
        if is_blank(cs) or fold(cs) in ("--", "-", "n/a", "na"):
            continue                                       # coder left it to EM-DAT
        if short in _SIMPLE:
            ev = emdat_get(row, _SIMPLE[short]); same = _same(cs, ev)
        elif short in _YESNO:
            ev = emdat_get(row, _YESNO[short])
            same = ev is not None and (yes_no_word(cs) or fold(cs)) == (yes_no_word(ev) or fold(ev))
        elif short in _MONEY:
            ev = [emdat_get(row, f) for f in _MONEY[short]]
            same = any(e is not None and _same(cs, e) for e in ev)
            ev = {"unadjusted": ev[0], "adjusted": ev[1]}
        elif short == "disaster_dates":
            ev = _date_text(row); same = dates_agree(cs, row)
        elif short == "disaster_subtype":
            ev = " / ".join(str(x) for x in (emdat_get(row, "subtype"), emdat_get(row, "subsubtype")) if x) or None
            same = ev is not None and any(fold(cs) == fold(x) for x in [ev, *ev.split(" / ")])
        else:                                              # lat_long: compare the numbers
            ev = f"{emdat_get(row, 'lat')}, {emdat_get(row, 'lon')}"
            nums = lambda s: [float(x) for x in re.findall(r"-?\d+(?:\.\d+)?", str(s))]
            a, b = nums(cs), nums(ev)
            same = len(a) == len(b) == 2 and all(abs(x - y) < 0.01 for x, y in zip(a, b))
        if not same:
            diffs.append({"variable": SHORT2KEY[short], "codesheet": cs, "emdat": ev,
                          "coder_may_revise": short in CODER_MAY_REVISE})
    return diffs

# ---------- the file name as a second check: "Bangladesh1988JuneFloodCodesheet.docx" ----------
FILE_TYPE_WORDS = {
    r"flood|inundation|monsoon": {"flood"},
    r"storm|cyclone|typhoon|hurricane|tornado|surge|nor.?wester|hail|blizzard": {"storm", "extreme temperature"},
    r"cold|heat|winter|frost": {"extreme temperature", "storm"},
    r"earthquake|quake|tremor|tsunami": {"earthquake"},
    r"landslide|mudslide|avalanche|mass ?movement": {"landslide", "mass movement (wet)", "mass movement (dry)"},
    r"drought|famine": {"drought", "complex disasters"},
    r"volcan|eruption": {"volcanic activity"},
    r"wildfire|forest ?fire|fire": {"wildfire"},
    r"epidemic|cholera|dengue|diarrh|outbreak|influenza|disease|covid": {"epidemic"},
}

# Reads the year, month and disaster type from a file name such as 'Bangladesh1988JuneFloodCodesheet.docx'.
def parse_file_name(name):
    stem = re.sub(r"(?i)\.docx?$", "", name)
    m = re.search(r"(?<!\d)((?:18|19|20)\d{2})(?!\d)", stem)
    if not m:
        return {}
    rest = stem[m.end():]
    mm = re.match(r"(?i)(january|february|march|april|may|june|july|august|september|october|november|"
                  r"december|sept|jan|feb|mar|apr|jun|jul|aug|sep|oct|nov|dec)", rest)
    month = _MONTHS.get(mm.group(1).lower()) if mm else None
    word = re.sub(r"(?i)code\s*sheet.*$", "", rest[mm.end():] if mm else rest).lower()
    types = set().union(*[t for pat, t in FILE_TYPE_WORDS.items() if re.search(pat, word)]) if word else set()
    return {"year": int(m.group(1)), "month": month, "type_word": word or None, "types": types or None}

# Start year of an EM-DAT disaster (taken from the Dis No if the Start Year cell is empty).
def _start(row):
    """Start year of an EM-DAT row (from the DisNo if Start Year is blank)."""
    sy = to_number(emdat_get(row, "sy"))
    event = dis_key(emdat_get(row, "disno"))[0]
    return sy or (int(event[:4]) if event else None)

# Lists the EM-DAT disasters that fit the file name's year, month and type (used when the Dis No on the
# sheet cannot be found).
def file_name_candidates(fn):
    if not fn.get("year"):
        return []
    out = []
    for row in EMDAT_BASE.to_dict("records"):
        if _start(row) != fn["year"]:
            continue
        if fn.get("month") and to_number(emdat_get(row, "sm")) not in (fn["month"], None):
            continue
        if fn.get("types") and fold(emdat_get(row, "type") or "") not in fn["types"]:
            continue
        out.append({"sheet": EMDAT_BASE_SHEET, "row": row})
    return out

# Finds this code sheet's EM-DAT disaster (by Dis No, or by the file name as a fallback) and stores its key
# fields in the JSON. Flags signs of a wrong Dis No: the file name, dates, type or country don't match EM-DAT.
def emdat_record(flat, source_path):
    """-> (EMDAT_Record block for the JSON, review reasons)."""
    sheet_disno = flat.get(SHORT2KEY["emdat_disno"])
    rec, how = find_emdat(sheet_disno)
    fn, reasons = parse_file_name(_name(source_path)), []
    event, iso = dis_key(sheet_disno)
    if rec is None and event and event.endswith("-0000"):
        reasons.append(f"not in EM-DAT ({disno_text(event, iso or HOME_ISO)}): a disaster found by the lab — "
                       f"the table fills its EM-DAT columns from the code sheet; check them")
    elif rec is None:
        cands = file_name_candidates(fn)
        if len(cands) == 1:
            rec, sheet_how = cands[0], how
            how = f"by the file name (DisNo on the sheet: {sheet_how})"
            reasons.append(f"EM-DAT: matched by the file name to {emdat_get(rec['row'], 'disno')} ({sheet_how} on the "
                           f"sheet) — confirm; if it is wrong, correct EMDAT_Record['Dis No']")
        else:
            reasons.append(f"EM-DAT: {how}" + (f"; the file name fits {len(cands)} EM-DAT disasters: "
                           f"{[emdat_get(c['row'], 'disno') for c in cands[:6]]}" if cands else ""))
    elif how != "exact":
        reasons.append(f"EM-DAT: DisNo {sheet_disno!r} {how}")
    block = {"Dis No": disno_text(event, iso or HOME_ISO) if event and rec is None and event.endswith("-0000") else None,
             "match": how, "emdat_file": _name(EMDAT_PATH), "emdat_sheet": None, "in_table_baseline": None,
             "key_fields": {}, "codesheet_differs": [],
             "note": ("The table uses this 'Dis No' as the disaster's ID: correct it here if the match is wrong. "
                      "EM-DAT values are used for the underlined variables, except dates, deaths, total affected "
                      "and declaration, where a differing code-sheet value replaces EM-DAT.")}
    if rec is None:
        return block, reasons
    row = rec["row"]
    disno = emdat_get(row, "disno")
    e2, i2 = dis_key(disno)
    block.update({"Dis No": disno_text(e2, i2 or str(emdat_get(row, "iso") or "").upper()),
                  "emdat_sheet": rec["sheet"], "in_table_baseline": rec["sheet"] == EMDAT_BASE_SHEET,
                  "key_fields": {emdat_col(row.keys(), f): emdat_get(row, f) for f in EMDAT_SNAPSHOT
                                 if emdat_col(row.keys(), f)}})
    if rec["sheet"] != EMDAT_BASE_SHEET:
        reasons.append(f"EM-DAT: {disno} ({emdat_get(row, 'type')}) is on sheet {rec['sheet']!r}, not on the baseline "
                       f"sheet {EMDAT_BASE_SHEET!r} — " + ("it will be left out of the table (ONLY_BASELINE_SHEET = True)"
                       if ONLY_BASELINE_SHEET else "the table adds it as an extra row"))
    # file-name check (year, month, disaster type)
    sy, sm, etype = _start(row), to_number(emdat_get(row, "sm")), fold(emdat_get(row, "type") or "")
    if fn.get("year") and sy and fn["year"] != sy:
        reasons.append(f"file name says {fn['year']} but EM-DAT {disno} starts in {sy} — right DisNo?")
    elif fn.get("month") and sm and fn["month"] != sm:
        reasons.append(f"file name says month {fn['month']} but EM-DAT {disno} starts in month {sm} — right DisNo?")
    if fn.get("types") and etype and etype not in fn["types"]:
        reasons.append(f"file name says {fn['type_word']!r} but EM-DAT {disno} is a {emdat_get(row, 'type')} — right DisNo?")
    # coder vs EM-DAT
    diffs = compare_with_emdat(flat, row)
    block["codesheet_differs"] = diffs
    strong = [d["variable"] for d in diffs if not d["coder_may_revise"] and d["emdat"] not in (None, {})
              and d["variable"].split("_", 1)[1] not in _FUZZY]
    if any(v.endswith("disaster_type") for v in strong) or len(strong) >= EMDAT_MISMATCH_REVIEW:
        reasons.append(f"EM-DAT: {len(strong)} field(s) differ from the sheet {strong[:6]} — right DisNo?")
    p = parse_dates(flat.get(SHORT2KEY["disaster_dates"]))
    if p["start"] and sy and abs(p["start"][0] - sy) > 1:
        reasons.append(f"dates on the sheet start in {p['start'][0]} but EM-DAT {disno} starts in {sy} — typo?")
    sheet_country, em_country = flat.get(SHORT2KEY["country_name"]), emdat_get(row, "country")
    if sheet_country and em_country and fold(sheet_country) != fold(em_country):
        reasons.append(f"EM-DAT: country on sheet {sheet_country!r} but EM-DAT says {em_country!r} — right DisNo?")
    return block, reasons

# ---------- nest the flat values into the JSON layout ----------
TABLE_OVERRIDES_HELP = ('Add "variable": value pairs to change a value in the Excel table only, e.g. '
                        '"reliefgoods": 2 (this also resolves the Review item for that variable). '
                        '"needs_review": 0 accepts all remaining Review items for this disaster. '
                        'Applied last when the table is built; keys starting with _ are ignored.')

# Puts everything into the final JSON layout: the template sections in order, then the coder name, notes,
# EM-DAT record, an empty Table_Overrides block for reviewers, and the extraction details.
def assemble(flat, info, rw, emdat_block, overrides=None):
    sections = {}
    for f in FIELDS:
        sections.setdefault(f["section"], {})[f["key"]] = flat[f["key"]]
    doc = {}
    for root in ROOT_ORDER:
        if root in sections:              doc[root] = sections[root]
        elif root in RELIEFWEB_BLOCKS:    doc[root] = rw.get(root)
        elif root == "EMDAT_Record":      doc[root] = emdat_block
        elif root == "Table_Overrides":   doc[root] = overrides or {"_help": TABLE_OVERRIDES_HELP}
        elif root == "Extraction_Info":   doc[root] = info
        else:                             doc[root] = flat[root]        # Coder_Name, Notes
    return doc

print("post-processing, checks and EM-DAT matching ready")

post-processing, checks and EM-DAT matching ready


### 12. Extraction loop (one file at a time, resumable)
Processes one Word file at a time: read it, send it to GPT, check the answer, match it to EM-DAT and save codesheet_<DisNo>.json in Data Files/JSONs. A log of finished files (_extraction_manifest.json) is saved after each file, so a stopped run can be restarted and unchanged files are skipped. A JSON that a person has marked as reviewed is never overwritten.

In [ ]:
MANIFEST_PATH = storage.join(JSON_FOLDER, "_extraction_manifest.json")
PRESENT = set()

def utc_now():
    return datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

def read_json(path):
    return json.loads(storage.read_bytes(path).decode("utf-8-sig"))

def write_json(path, obj):
    """Multi-line, indented, UTF-8 (curly quotes and dashes stay readable)."""
    storage.write_bytes(path, (json.dumps(obj, ensure_ascii=False, indent=2, default=str) + "\n").encode("utf-8"))

def load_manifest():
    return read_json(MANIFEST_PATH) if storage.exists(MANIFEST_PATH) else {}

# Makes text safe to use in a file name by replacing spaces and unusual characters with '_'.
def _safe(s):
    return re.sub(r"[^\w.\-]+", "_", str(s)).strip("._")[:80] or "file"

# True if a person has marked this JSON as reviewed ("human_reviewed": true).
def _reviewed(doc):
    return bool(isinstance(doc, dict) and (doc.get("Extraction_Info") or {}).get("human_reviewed"))

DISNO_RE = re.compile(r"\d{4}-\d{4}(-[A-Z]{3})?")

# Handles one Word file: skips it if unchanged since a successful run; otherwise reads it, asks GPT, checks
# the answer, matches EM-DAT and saves codesheet_<DisNo>.json. Keeps reviewers' Table_Overrides, never
# overwrites a reviewed JSON, and flags two Word files that claim the same Dis No.
def process_one(path, manifest):
    data = storage.read_bytes(path)
    sha = hashlib.sha256(data).hexdigest()
    prev = manifest.get(path) or {}
    if prev.get("sha256") == sha and prev.get("status") == "ok" and not REDO_UNCHANGED:
        return {**prev, "action": "skipped (unchanged)"}
    stem = _safe(_name(path).rsplit(".", 1)[0])
    rec = {"source": path, "sha256": sha, "extracted_at_utc": utc_now(), "output": None,
           "disno": None, "needs_human_review": True, "review_reasons": [], "usage": {}}

    outline, lines = read_codesheet(data)
    if not outline:
        return {**rec, "status": "empty_document"}
    api = call_model(outline)
    rec["usage"] = api.get("usage") or {}
    obj = parse_json(api.get("text")) if api["status"] == "ok" else None
    if api["status"] != "ok":                     status = "api_error"
    elif api.get("refusal"):                      status = "refusal"
    elif api.get("finish") in ("max_output_tokens", "length", "incomplete"): status = "truncated"
    elif not isinstance(obj, dict):               status = "invalid_json"
    else:                                         status = "ok"
    if status != "ok":                            # nothing written; the next run retries this file
        return {**rec, "status": status, "error": api.get("error") or api.get("refusal") or api.get("finish")}

    flat, notes = coerce(obj)                                   # the coder's values, as typed
    emdat_block, emdat_reasons = emdat_record(flat, path)       # which EM-DAT disaster this is
    reasons = review_checks(flat, lines, outline) + emdat_reasons + [f"coercion: {n}" for n in notes]
    sheet_disno = str(flat[SHORT2KEY["emdat_disno"]] or "")
    disno = emdat_block.get("Dis No") or (sheet_disno if DISNO_RE.fullmatch(sheet_disno) else None)
    ok_id = bool(disno and DISNO_RE.fullmatch(disno))
    if not ok_id:
        out_name = f"codesheet_UNKNOWN_{stem}.json"
    elif "-0000" in disno:                        # several lab-found disasters can share YYYY-0000-ISO
        out_name = f"codesheet_{disno}__{stem}.json"
    else:
        out_name = f"codesheet_{disno}.json"
    owner = next((src for src, m in manifest.items() if ok_id and "-0000" not in disno and src != path
                  and src in PRESENT and m.get("status") == "ok" and m.get("disno") == disno), None)
    if owner:                                     # two Word files claim the same DisNo: keep both, flag this one
        reasons.append(f"duplicate DisNo: also in {_name(owner)} — the table uses only one of them")
        out_name = f"codesheet_{disno}__{stem}.json"
    out_path = storage.join(JSON_FOLDER, out_name)

    overrides = None
    if storage.exists(out_path):
        try:
            old = read_json(out_path)
        except Exception:
            old = None
        if _reviewed(old):                        # never overwrite a JSON a person has already reviewed
            out_path = storage.join(JSON_FOLDER, out_name.replace(".json", f"__reextracted_{utc_now()[:10]}.json"))
            reasons.append("a human-reviewed JSON already exists; this re-extraction was saved beside it")
        elif isinstance(old, dict) and any(not k.startswith("_") for k in (old.get("Table_Overrides") or {})):
            overrides = old["Table_Overrides"]    # keep a person's table overrides across re-extractions
    old_out = prev.get("output")                  # this Word file's previous JSON under another name
    if old_out and old_out != out_path and storage.exists(old_out):
        try:
            old_reviewed = _reviewed(read_json(old_out))
        except Exception:
            old_reviewed = False
        if old_reviewed:
            reasons.append(f"the earlier JSON {_name(old_out)} was human-reviewed and is kept — delete it "
                           f"if this one replaces it")
        else:
            storage.move(old_out, storage.join(JSON_FOLDER, f"_superseded/{_name(old_out)}"))

    info = {
        "source_file": path, "source_sha256": sha, "extracted_at_utc": rec["extracted_at_utc"],
        "model": MODEL, "reasoning_effort": REASONING_EFFORT, "strict_schema": STRICT_ON,
        "prompt_version": PROMPT_VER, "prompt_hash": PROMPT_HASH,
        "is_fewshot_example": sha in FEWSHOT_HASHES,
        "needs_human_review": bool(reasons), "review_reasons": reasons,
        "unmapped_items": flat["Unmapped_Items"], "model_comments": flat["Model_Comments"],
        "human_reviewed": False, "reviewer": None, "review_date": None,
    }
    write_json(out_path, assemble(flat, info, reliefweb_blocks(disno), emdat_block, overrides))
    return {**rec, "status": "ok", "output": out_path, "disno": disno,
            "needs_human_review": bool(reasons), "review_reasons": reasons}

# Runs process_one on a list of files, saving the log after each one so the run can be stopped and resumed.
# Prints one status line per file and returns a summary table.
def run_extraction(files):
    global PRESENT
    PRESENT = set(storage.list_files(CODESHEET_FOLDER, RECURSIVE))    # Word files that exist right now
    manifest, log = load_manifest(), []
    for path in tqdm(files, desc="code sheets"):
        try:
            rec = process_one(path, manifest)
        except Exception as e:
            rec = {"source": path, "status": "error", "error": f"{type(e).__name__}: {e}",
                   "needs_human_review": True, "review_reasons": [], "usage": {}}
        if not str(rec.get("action", "")).startswith("skipped"):
            manifest[path] = {k: v for k, v in rec.items() if k != "action"}
            write_json(MANIFEST_PATH, manifest)       # saved after every file -> safe to stop and resume
        log.append(rec)
        u = rec.get("usage") or {}
        tqdm.write(f"[{_name(path)[:45]:<45}] {rec.get('action') or rec['status']:<20} "
                   f"{rec.get('disno') or '':<14} review={'YES' if rec.get('needs_human_review') else 'no'} "
                   f"in={u.get('input_tokens')} cached={u.get('cached_tokens')} out={u.get('output_tokens')}"
                   + (f" | {rec.get('error')}" if rec.get("error") else ""))
    return pd.DataFrame(log)

print("extraction loop ready")

extraction loop ready


### 13. Pilot run (compare these JSONs with their Word files before the full run)

In [ ]:
pilot = run_extraction(DOCS[:PILOT_N])

for _, r in pilot.iterrows():
    out = r.get("output")
    print(f"\n── {_name(r['source'])}: {r['action'] if isinstance(r.get('action'), str) else r['status']} -> "
          f"{_name(out) if isinstance(out, str) else '(nothing written) ' + str(r.get('error') or '')}")
    for reason in (r.get("review_reasons") if isinstance(r.get("review_reasons"), list) else []):
        print("   ⚠️", reason)

first_ok = next((r["output"] for _, r in pilot.iterrows() if isinstance(r.get("output"), str)), None)
if first_ok:
    print(f"\n── {_name(first_ok)} (compare this against the Word file) ──")
    print(json.dumps(read_json(first_ok), ensure_ascii=False, indent=2)[:6000])

code sheets:   0%|          | 0/3 [00:00<?, ?it/s]

[Bangladesh1904NovemberCycloneCodesheet.docx  ] ok                   1904-0003-BGD  review=no in=6432 cached=0 out=802
[Bangladesh1911MarchCycloneCodesheet.docx     ] ok                   1911-0004-BGD  review=YES in=6442 cached=5652 out=670
[Bangladesh1923MayCycloneCodesheet.docx       ] ok                   1923-0003-BGD  review=no in=6368 cached=5652 out=600

── Bangladesh1904NovemberCycloneCodesheet.docx: ok -> codesheet_1904-0003-BGD.json

── Bangladesh1911MarchCycloneCodesheet.docx: ok -> codesheet_1911-0004-BGD.json
   ⚠️ source text not in output (1): ['No further details about exact date in EMDAT or on the internet']
   ⚠️ file name says month 3 but EM-DAT 1911-0004-BGD starts in month 4 — right DisNo?

── Bangladesh1923MayCycloneCodesheet.docx: ok -> codesheet_1923-0003-BGD.json

── codesheet_1904-0003-BGD.json (compare this against the Word file) ──
{
  "General Information": {
    "01_country_name": "Bangladesh",
    "02_cow_country_code": 771,
    "03_emdat_disno": "1904-0

### 14. Full run

In [ ]:
run_log = run_extraction(DOCS)      # already-extracted, unchanged files are skipped, so re-running resumes
print("\n", run_log["status"].value_counts(dropna=False).to_string())

code sheets:   0%|          | 0/147 [00:00<?, ?it/s]

[Bangladesh1904NovemberCycloneCodesheet.docx  ] skipped (unchanged)  1904-0003-BGD  review=no in=6432 cached=0 out=802
[Bangladesh1911MarchCycloneCodesheet.docx     ] skipped (unchanged)  1911-0004-BGD  review=YES in=6442 cached=5652 out=670
[Bangladesh1923MayCycloneCodesheet.docx       ] skipped (unchanged)  1923-0003-BGD  review=no in=6368 cached=5652 out=600
[Bangladesh1941MayCycloneCodesheet.docx       ] ok                   1941-0003-BGD  review=no in=6354 cached=0 out=464
[Bangladesh1942OctoberCycloneCodesheet.docx   ] ok                   1942-0008-BGD  review=no in=6356 cached=5652 out=609
[Bangladesh1943DroughtCodesheet.docx          ] ok                   1943-9002-BGD  review=no in=6524 cached=5652 out=1013
[Bangladesh1947OctoberCycloneCodesheet.docx   ] ok                   1947-0012-BGD  review=no in=6332 cached=5652 out=571
[Bangladesh1958MayCyloneCodesheet.docx        ] ok                   1958-0008-BGD  review=no in=6416 cached=5652 out=650
[Bangladesh1958OctoberCyclon

### 15. Quality-control report & review queue

In [ ]:
m = pd.DataFrame(load_manifest().values())
for c in ("usage", "needs_human_review", "review_reasons", "disno", "output", "error", "status"):
    if c not in m:
        m[c] = None
print(f"── {len(m)} Word file(s) in the manifest ──")
print(m["status"].value_counts().to_string())

u = pd.json_normalize(m["usage"].apply(lambda x: x if isinstance(x, dict) else {}))
tot = {c: pd.to_numeric(u.get(c), errors="coerce").fillna(0).sum() for c in
       ("input_tokens", "cached_tokens", "output_tokens", "reasoning_tokens")}
cost = ((tot["input_tokens"] - tot["cached_tokens"]) * PRICE["input"] + tot["cached_tokens"] * PRICE["cached_input"]
        + tot["output_tokens"] * PRICE["output"]) / 1e6
print(f"\ntokens in {tot['input_tokens']:,.0f} (cached {tot['cached_tokens']:,.0f}) | out {tot['output_tokens']:,.0f} "
      f"(reasoning {tot['reasoning_tokens']:,.0f}) | ≈ ${cost:,.2f} for the latest extraction of each file")

queue = m[(m["status"] != "ok") | (m["needs_human_review"] == True)].copy()
queue["file"] = queue["source"].map(_name)
queue["json"] = queue["output"].map(lambda p: _name(p) if isinstance(p, str) else None)
queue["reasons"] = queue["review_reasons"].apply(lambda x: "\n".join(x) if isinstance(x, list) else "")
queue = queue[["file", "status", "disno", "json", "reasons", "error", "source"]].sort_values(["status", "file"])
print(f"\n{len(queue)} file(s) need a look ({(m['status'] != 'ok').sum()} failed, the rest flagged)")

buf = io.BytesIO()
queue.to_excel(buf, index=False)
REVIEW_QUEUE = storage.join(JSON_FOLDER, "_review_queue.xlsx")
storage.write_bytes(REVIEW_QUEUE, buf.getvalue())
print("review queue ->", REVIEW_QUEUE)
queue.head(20)

── 147 Word file(s) in the manifest ──
status
ok    147

tokens in 965,386 (cached 819,540) | out 128,173 (reasoning 9,232) | ≈ $1.66 for the latest extraction of each file

21 file(s) need a look (0 failed, the rest flagged)
review queue -> /DisastersProject/Bangladesh/Data Files/JSONs/_review_queue.xlsx


,file,status,disno,json,reasons,error,source
1,Bangladesh1911MarchCycloneCodesheet.docx,ok,1911-0004-BGD,codesheet_1911-0004-BGD.json,source text not in output (1): ['No further de...,None,/DisastersProject/Bangladesh/Codesheets/Bangla...
18,Bangladesh1964JulyFloodCodesheet.docx,ok,1964-0039-BGD,codesheet_1964-0039-BGD.json,not verbatim in source: ['Notes'],None,/DisastersProject/Bangladesh/Codesheets/Bangla...
37,Bangladesh1974OctoberDroughtCodesheet.docx,ok,1974-9049-BGD,codesheet_1974-9049-BGD.json,model comments: ['The Officials visited answer...,None,/DisastersProject/Bangladesh/Codesheets/Bangla...
46,Bangladesh1982JulyDroughtCodesheet.docx,ok,1982-9350-BGD,codesheet_1982-9350-BGD.json,file name says 1982 but EM-DAT 1982-9350-BGD s...,None,/DisastersProject/Bangladesh/Codesheets/Bangla...
55,Bangladesh1987JuneCycloneCodesheet.docx,ok,1987-0110-BGD,codesheet_1987-0110-BGD.json,unexpected coded value 31_gov_regulated_outsid...,None,/DisastersProject/Bangladesh/Codesheets/Bangla...
59,Bangladesh1988NovemberStormCodesheet.docx,ok,1988-0518-BGD,codesheet_1988-0518-BGD.json,"model comments: ['NGO actors is answered as ""-...",None,/DisastersProject/Bangladesh/Codesheets/Bangla...
68,Bangladesh1991MayCycloneCodesheet.docx,ok,1991-0120-BGD,codesheet_1991-0120-BGD.json,non-numeric 12_aid_contribution_total_kusd: '$...,None,/DisastersProject/Bangladesh/Codesheets/Bangla...
69,Bangladesh1991MayFloodCodesheet.docx,ok,1991-0764-BGD,codesheet_1991-0764-BGD.json,EM-DAT: 3 field(s) differ from the sheet ['02_...,None,/DisastersProject/Bangladesh/Codesheets/Bangla...
75,Bangladesh1993JulyFloodCodesheet.docx,ok,1993-0045-BGD,codesheet_1993-0045-BGD.json,non-numeric 12_aid_contribution_total_kusd: 'No',None,/DisastersProject/Bangladesh/Codesheets/Bangla...
78,Bangladesh1994AprilFloodCodesheet.docx,ok,1994-0754-BGD,codesheet_1994-0754-BGD.json,model comments: ['Disaster Dates contains the ...,None,/DisastersProject/Bangladesh/Codesheets/Bangla...


---
# Part B: Build the disaster-level Excel table from the JSONs

The next three cells are identical to the *JSON → Excel* notebook. We can use that notebook to rebuild the table after people have reviewed or corrected JSONs, without re-running GPT. The table is always rebuilt from **all** JSONs in `Data Files/JSONs`, so corrections made in the JSONs carry through.

### 16. Table coding rules (edit here to change how code sheets are coded)
ll the rules for turning code-sheet answers into numbers, following "Instructions for entering DMV data (AI version)": column names, -9 for missing, 0/1 for No/Yes, and 0–3 for None/Low/Medium/High. It also holds the keyword lists that decide which government actors count as military, police or civil defence, and the Correlates of War (COW) codes used for other states. Change coding decisions here, not in the cells below.

In [ ]:
TABLE_RULES_VERSION = "2026.10.07"
MISSING   = -9      # "Missing/case has no information"
NOT_CODED = None    # the new variables for EM-DAT disasters that have no code sheet yet (None = empty cell)

# The three exceptions in the instructions: where the coder's value differs from EM-DAT, the coder's value is used.
APPLY_CODER_REVISIONS = {"dates": True, "deaths": True, "affected": True, "declaration": True}
MAX_YEAR_CHANGE = 1                 # a revised start year further than this from EM-DAT's is treated as a typo
                                    # (e.g. 1940 for a 1950 earthquake): EM-DAT dates are kept and it is flagged
RECATEGORIZE_STATE_AGENCIES = True  # e.g. USAID listed as an NGO is counted as the United States (logged)

# The new variables, in the order of the instructions:  (column name, code-sheet question, how it is coded)
#   level : Missing -9 / None 0 / Low 1 / Medium 2 / High 3      yesno : Missing -9 / No 0 / Yes 1
#   count : number of separate entries listed (-9 if nothing listed, 0 if the coder wrote "None")
#   military / police / civildef : 1 if any actor listed in question 24 matches the keywords below
#   states: number of other states in question 28, plus one COW-code column per state (state1_cow, ...)
#   text  : copied as written
# Rename a column here to change it everywhere (Peru's table calls govinfo "govtinfo" and igos "ios").
TABLE_VARIABLES = [
    ("cname",             "country_name",                   "text"),
    ("ccode",             "cow_country_code",               "ccode"),
    ("emdat",             "emdat_disno",                    "disno"),
    ("nonemdatsource",    "non_emdat_source",               "text"),
    ("description",       "brief_description",              "text"),
    ("govinfo",           "gov_info_level",                 "level"),      # question 20
    ("shelters",          "shelters_provided",              "yesno"),      # 21
    ("tourists",          "tourist_area",                   "yesno"),      # 22
    ("reliefgoods",       "relief_goods_level",             "level"),      # 23
    ("govtactors",        "gov_actors",                     "count"),      # 24
    ("military",          "gov_actors",                     "military"),   # 24
    ("police",            "gov_actors",                     "police"),     # 24
    ("civildef",          "gov_actors",                     "civildef"),   # 24
    ("leaders",           "officials_visited",              "count"),      # 25
    ("ngos",              "ngo_actors",                     "count"),      # 26
    ("igos",              "io_actors",                      "count"),      # 27
    ("states",            "other_states",                   "states"),     # 28
    ("requestaid",        "gov_requested_outside_aid",      "yesno"),      # 30
    ("restrictaid",       "gov_regulated_outside_aid",      "yesno"),      # 31
    ("responsefailure",   "gov_response_failure",           "yesno"),      # 32
    ("coordinationlevel", "gov_outside_coordination_level", "level"),      # 33
    ("govtcoordination",  "gov_coordination_of_aid_actors", "level"),      # 34
    ("govtngomeetings",   "gov_led_meetings",               "yesno"),      # 35
    ("tempevac",          "temporary_evacuations",          "yesno"),      # 36
    ("permevac",          "permanent_relocation",           "yesno"),      # 37
    ("moverestrict",      "movement_restrictions",          "yesno"),      # 38
    ("returnrestrict",    "return_home_restrictions",       "yesno"),      # 39
    ("reconstresources",  "reconstruction_resources_level", "level"),      # 40
    ("reconstlevel",      "reconstruction_funding_level",   "text"),       # 41 (Peru's "Reconst Level")
]
STATE_COLUMN = "state{}_cow"
# bookkeeping columns at the far right
INFO_COLUMNS = ["ocharep", "codesheet", "coder", "codesheet_notes", "emdat_changes", "needs_review", "json_file"]

# ---------- reading the coders' answers ----------
LEVEL_WORDS = {"high": 3, "h": 3, "medium": 2, "med": 2, "m": 2, "moderate": 2, "low": 1, "l": 1,
               "none": 0, "no": 0, "n": 0}
YESNO_WORDS = {"yes": 1, "y": 1, "no": 0, "n": 0, "none": 0}
MISSING_ANSWERS = ["--", "-", "n/a", "na", "n.a.", "nan", "unknown", "missing", "no information", "no info",
                   "not available", "not applicable", "not found", "unclear", "?", "-9", "no data", "no stories",
                   "not mentioned", "no mention", "not reported", "not known", "not specified", "not stated"]
NONE_ENTRIES = ["none", "no", "0", "nil", "none reported", "none found", "none listed", "none mentioned", "no one"]

# ---------- question 24: which government actors are military / police / civil defence ----------
# "words" match anywhere (any case); "acronyms" match whole, upper-case words only; "exclude" vetoes a match.
# civildef is the country's civil defence / protection / national disaster-management body (INDECI for Peru,
# FEMA for the US, OCD for the Philippines). For Bangladesh the list below counts the Disaster Management Bureau
# (1993-2012), the Department of Disaster Management (2012-), the disaster-management / relief ministries
# (Ministry of Relief and Rehabilitation in the 1980s-90s, later Food and Disaster Management, now Disaster
# Management and Relief) and Fire Service & Civil Defence — PLEASE CONFIRM this is how the lab wants it coded.
ACTOR_KEYWORDS = {
    "military": {"words": ["army", "navy", "naval", "air force", "airforce", "military", "armed forces",
                           "armed force", "troops", "soldier", "coast guard", "coastguard", "marines", "marine corps",
                           "national guard", "defence force", "defense force", "border guard", "bangladesh rifles",
                           "paramilitary", "ministry of defence", "ministry of defense", "armed forces division"],
                 "acronyms": ["BGB", "BDR", "AFD"],
                 "exclude": ["salvation army"]},
    "police":   {"words": ["police", "constabulary", "gendarmerie", "rapid action battalion"],
                 "acronyms": ["RAB"], "exclude": []},
    "civildef": {"words": ["civil defense", "civil defence", "civil protection", "defensa civil",
                           "disaster management", "relief and rehabilitation"],
                 "acronyms": ["INDECI", "FEMA", "OCD", "NDRRMC", "DDM", "DMB", "MoDMR", "MDMR", "FSCD"],
                 "exclude": []},
}

# ---------- question 28: Correlates of War state codes (COW State System Membership, states2016) ----------
# code:name:first year:last year. Year rules: Germany is 260 (German Federal Republic) for 1955-1989 and 255
# otherwise; Yemen before 1990 is 678 (Yemen Arab Republic); Czech Republic/Slovakia before 1993 are 315.
COW_STATES = """
2:United States of America:1816:2016|20:Canada:1920:2016|31:Bahamas:1973:2016|40:Cuba:1902:2016
41:Haiti:1859:2016|42:Dominican Republic:1894:2016|51:Jamaica:1962:2016|52:Trinidad and Tobago:1962:2016
53:Barbados:1966:2016|54:Dominica:1978:2016|55:Grenada:1974:2016|56:St. Lucia:1979:2016
57:St. Vincent and the Grenadines:1979:2016|58:Antigua & Barbuda:1981:2016|60:St. Kitts and Nevis:1983:2016
70:Mexico:1831:2016|80:Belize:1981:2016|90:Guatemala:1868:2016|91:Honduras:1899:2016|92:El Salvador:1875:2016
93:Nicaragua:1900:2016|94:Costa Rica:1920:2016|95:Panama:1903:2016|100:Colombia:1831:2016
101:Venezuela:1841:2016|110:Guyana:1966:2016|115:Suriname:1975:2016|130:Ecuador:1854:2016|135:Peru:1839:2016
140:Brazil:1822:2016|145:Bolivia:1848:2016|150:Paraguay:1846:2016|155:Chile:1839:2016|160:Argentina:1841:2016
165:Uruguay:1882:2016|200:United Kingdom:1816:2016|205:Ireland:1922:2016|210:Netherlands:1816:2016
211:Belgium:1830:2016|212:Luxembourg:1920:2016|220:France:1816:2016|221:Monaco:1993:2016
223:Liechtenstein:1990:2016|225:Switzerland:1816:2016|230:Spain:1816:2016|232:Andorra:1993:2016
235:Portugal:1816:2016|240:Hanover:1837:1866|245:Bavaria:1816:1871|255:Germany:1816:2016
260:German Federal Republic:1955:1989|265:German Democratic Republic:1954:1990|267:Baden:1816:1871
269:Saxony:1816:1867|271:Wuerttemburg:1816:1871|273:Hesse Electoral:1816:1866|275:Hesse Grand Ducal:1816:1867
280:Mecklenburg Schwerin:1843:1867|290:Poland:1918:2016|300:Austria-Hungary:1816:1918|305:Austria:1919:2016
310:Hungary:1918:2016|315:Czechoslovakia:1918:1992|316:Czech Republic:1993:2016|317:Slovakia:1993:2016
325:Italy:1816:2016|327:Papal States:1816:1860|329:Two Sicilies:1816:1861|331:San Marino:1992:2016
332:Modena:1842:1860|335:Parma:1851:1860|337:Tuscany:1816:1860|338:Malta:1964:2016|339:Albania:1914:2016
341:Montenegro:2006:2016|343:Macedonia:1993:2016|344:Croatia:1992:2016|345:Yugoslavia:1878:2016
346:Bosnia and Herzegovina:1992:2016|347:Kosovo:2008:2016|349:Slovenia:1992:2016|350:Greece:1828:2016
352:Cyprus:1960:2016|355:Bulgaria:1908:2016|359:Moldova:1991:2016|360:Romania:1878:2016|365:Russia:1816:2016
366:Estonia:1918:2016|367:Latvia:1918:2016|368:Lithuania:1918:2016|369:Ukraine:1991:2016|370:Belarus:1991:2016
371:Armenia:1991:2016|372:Georgia:1991:2016|373:Azerbaijan:1991:2016|375:Finland:1917:2016|380:Sweden:1816:2016
385:Norway:1905:2016|390:Denmark:1816:2016|395:Iceland:1944:2016|402:Cape Verde:1975:2016
403:Sao Tome and Principe:1975:2016|404:Guinea-Bissau:1974:2016|411:Equatorial Guinea:1968:2016
420:Gambia:1965:2016|432:Mali:1960:2016|433:Senegal:1960:2016|434:Benin:1960:2016|435:Mauritania:1960:2016
436:Niger:1960:2016|437:Ivory Coast:1960:2016|438:Guinea:1958:2016|439:Burkina Faso:1960:2016
450:Liberia:1920:2016|451:Sierra Leone:1961:2016|452:Ghana:1957:2016|461:Togo:1960:2016|471:Cameroon:1960:2016
475:Nigeria:1960:2016|481:Gabon:1960:2016|482:Central African Republic:1960:2016|483:Chad:1960:2016
484:Congo:1960:2016|490:Democratic Republic of the Congo:1960:2016|500:Uganda:1962:2016|501:Kenya:1963:2016
510:Tanzania:1961:2016|511:Zanzibar:1963:1964|516:Burundi:1962:2016|517:Rwanda:1962:2016|520:Somalia:1960:2016
522:Djibouti:1977:2016|530:Ethiopia:1898:2016|531:Eritrea:1993:2016|540:Angola:1975:2016
541:Mozambique:1975:2016|551:Zambia:1964:2016|552:Zimbabwe:1965:2016|553:Malawi:1964:2016
560:South Africa:1920:2016|565:Namibia:1990:2016|570:Lesotho:1966:2016|571:Botswana:1966:2016
572:Swaziland:1968:2016|580:Madagascar:1960:2016|581:Comoros:1975:2016|590:Mauritius:1968:2016
591:Seychelles:1976:2016|600:Morocco:1847:2016|615:Algeria:1962:2016|616:Tunisia:1825:2016|620:Libya:1951:2016
625:Sudan:1956:2016|626:South Sudan:2011:2016|630:Iran:1855:2016|640:Turkey:1816:2016|645:Iraq:1932:2016
651:Egypt:1855:2016|652:Syria:1946:2016|660:Lebanon:1946:2016|663:Jordan:1946:2016|666:Israel:1948:2016
670:Saudi Arabia:1927:2016|678:Yemen Arab Republic:1926:1990|679:Yemen:1990:2016
680:Yemen People's Republic:1967:1990|690:Kuwait:1961:2016|692:Bahrain:1971:2016|694:Qatar:1971:2016
696:United Arab Emirates:1971:2016|698:Oman:1971:2016|700:Afghanistan:1919:2016|701:Turkmenistan:1991:2016
702:Tajikistan:1991:2016|703:Kyrgyzstan:1991:2016|704:Uzbekistan:1991:2016|705:Kazakhstan:1991:2016
710:China:1860:2016|712:Mongolia:1921:2016|713:Taiwan:1949:2016|730:Korea:1887:1905|731:North Korea:1948:2016
732:South Korea:1949:2016|740:Japan:1860:2016|750:India:1947:2016|760:Bhutan:1971:2016|770:Pakistan:1947:2016
771:Bangladesh:1971:2016|775:Myanmar:1948:2016|780:Sri Lanka:1948:2016|781:Maldives:1965:2016
790:Nepal:1920:2016|800:Thailand:1887:2016|811:Cambodia:1953:2016|812:Laos:1953:2016|816:Vietnam:1954:2016
817:Republic of Vietnam:1954:1975|820:Malaysia:1957:2016|830:Singapore:1965:2016|835:Brunei:1984:2016
840:Philippines:1946:2016|850:Indonesia:1949:2016|860:East Timor:2002:2016|900:Australia:1920:2016
910:Papua New Guinea:1975:2016|920:New Zealand:1920:2016|935:Vanuatu:1981:2016|940:Solomon Islands:1978:2016
946:Kiribati:1999:2016|947:Tuvalu:2000:2016|950:Fiji:1970:2016|955:Tonga:1999:2016|970:Nauru:1999:2016
983:Marshall Islands:1991:2016|986:Palau:1994:2016|987:Federated States of Micronesia:1991:2016
990:Samoa:1976:2016
"""
COW_ALIASES = {   # other names people write -> COW code (lower case, accents and punctuation ignored)
    "united states": 2, "usa": 2, "us": 2, "america": 2, "american": 2, "uk": 200, "britain": 200,
    "great britain": 200, "england": 200, "british": 200, "english": 200, "west germany": 260,
    "federal republic of germany": 260, "east germany": 265, "cote d ivoire": 437, "drc": 490, "dr congo": 490,
    "zaire": 490, "congo kinshasa": 490, "republic of the congo": 484, "congo brazzaville": 484, "burma": 775,
    "ceylon": 780, "viet nam": 816, "south vietnam": 817, "dprk": 731, "democratic people s republic of korea": 731,
    "korea": 732, "republic of korea": 732, "rok": 732, "russian federation": 365, "ussr": 365,
    "soviet union": 365, "soviet": 365, "czechia": 316, "north macedonia": 343, "eswatini": 572,
    "cabo verde": 402, "timor leste": 860, "north yemen": 678, "south yemen": 680, "persia": 630, "turkiye": 640,
    "holland": 210, "the netherlands": 210, "saint lucia": 56, "saint vincent and the grenadines": 57,
    "saint kitts and nevis": 60, "antigua and barbuda": 58, "bosnia": 346, "micronesia": 987,
    "chinese taipei": 713, "prc": 710, "people s republic of china": 710, "uae": 696, "emirates": 696,
    "saudi": 670, "ksa": 670, "png": 910, "the gambia": 420, "the bahamas": 31, "brunei darussalam": 835,
    "lao pdr": 812, "lao": 812, "syrian arab republic": 652, "kyrgyz republic": 703, "slovak republic": 317,
    "serbia": 345, "trinidad": 52, "new zealand": 920,
    # demonyms (e.g. "Japanese Government", "Indian Air Force")
    "japanese": 740, "indian": 750, "pakistani": 770, "chinese": 710, "german": 255, "french": 220,
    "canadian": 20, "australian": 900, "swedish": 380, "norwegian": 385, "danish": 390, "dutch": 210,
    "swiss": 225, "kuwaiti": 690, "emirati": 696, "qatari": 694, "south korean": 732, "north korean": 731,
    "turkish": 640, "italian": 325, "spanish": 230, "russian": 365, "iranian": 630, "iraqi": 645,
    "belgian": 211, "finnish": 375, "austrian": 305, "irish": 205, "nepalese": 790, "nepali": 790,
    "sri lankan": 780, "thai": 800, "malaysian": 820, "indonesian": 850, "singaporean": 830, "filipino": 840,
    "philippine": 840, "bhutanese": 760, "burmese": 775, "egyptian": 651, "libyan": 620, "omani": 698,
    "bahraini": 692, "brazilian": 140, "mexican": 70, "cuban": 40, "venezuelan": 101, "chilean": 155,
    "argentine": 160, "argentinian": 160, "colombian": 100, "ecuadorian": 130, "bolivian": 145, "peruvian": 135,
    "israeli": 666, "greek": 350, "portuguese": 235, "polish": 290, "hungarian": 310, "czech": 316,
    "maldivian": 781, "afghan": 700, "jordanian": 663, "lebanese": 660, "syrian": 652, "algerian": 615,
    "moroccan": 600, "nigerian": 475, "south african": 560, "vietnamese": 816, "mongolian": 712,
    "taiwanese": 713, "ukrainian": 369, "romanian": 360, "bulgarian": 355, "luxembourg": 212,
}
# government aid agencies -> the state they belong to (counted as that state, also when listed as an NGO or IO)
STATE_AGENCIES = {
    "usaid": 2, "us aid": 2, "ofda": 2, "us agency for international development": 2,
    "united states agency for international development": 2, "dfid": 200, "department for international development": 200,
    "ukaid": 200, "uk aid": 200, "fcdo": 200, "jica": 740, "japan international cooperation agency": 740,
    "cida": 20, "canadian international development agency": 20, "global affairs canada": 20, "ausaid": 900,
    "sida": 380, "norad": 385, "danida": 390, "finnida": 375, "gtz": 255, "giz": 255, "kfw": 255, "thw": 255,
    "sdc": 225, "swiss agency for development and cooperation": 225, "koica": 732, "tika": 640, "afad": 640,
    "kuwait fund": 690, "saudi fund for development": 670, "ksrelief": 670, "aecid": 230, "irish aid": 205,
    "nzaid": 920,
}

print(f"coding rules {TABLE_RULES_VERSION}: {len(TABLE_VARIABLES)} variables")

coding rules 2026.10.07: 29 variables


### 17. Table builder (JSONs + EM-DAT → one row per disaster)
Defines how the table is built: start from every EM-DAT sheet-1 disaster sorted by Dis No, attach each disaster's JSON, use the coder's revised dates, deaths, total affected and declaration where they differ, and add the coded variables at the right edge. Disasters without a code sheet keep the new columns empty (codesheet = 0). It also writes the extra sheets: Review, Coding_Audit, Codesheet_Fields, Codebook and About.

In [ ]:
from openpyxl import Workbook
from openpyxl.comments import Comment
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter
from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE

# ---------- COW codes ----------
COW = {}                                   # code -> (name, first year, last year)
for _item in COW_STATES.replace("\n", "|").split("|"):
    if _item.strip():
        _c, _n, _a, _b = _item.strip().split(":")
        COW[int(_c)] = (_n, int(_a), int(_b))

# Simplifies a name before looking for country names: removes accents and punctuation, lower case,
# 'U.S.' -> 'united states', 'U.K.' -> 'united kingdom'.
def _fold_name(s):
    s = unicodedata.normalize("NFKD", str(s))
    s = "".join(ch for ch in s if not unicodedata.combining(ch)).lower().translate(_FOLD)
    s = re.sub(r"\bu\.\s?s\.(\s?a\.)?", " united states ", s)
    s = re.sub(r"\bu\.\s?k\.", " united kingdom ", s)
    s = re.sub(r"'s\b", "", s)
    return re.sub(r"[^a-z0-9]+", " ", s).strip()

_ALIAS = {_fold_name(n): c for c, (n, _, _) in COW.items()}
_ALIAS.update({_fold_name(a): c for a, c in COW_ALIASES.items()})
_AGENCY = {_fold_name(a): c for a, c in STATE_AGENCIES.items()}
_alt = lambda names: "|".join(sorted(map(re.escape, names), key=len, reverse=True))
_STATE_RE = re.compile(rf"(?<![a-z0-9])({_alt(list(_ALIAS) + list(_AGENCY))})(?![a-z0-9])")
_AGENCY_RE = re.compile(rf"(?<![a-z0-9])({_alt(_AGENCY)})(?![a-z0-9])")
_GOV = (r"(?:government|govt|embassy|consulate|high commission|ministry of foreign affairs|foreign ministry|"
        r"armed forces|military|army|navy|air force|marines|department of state|state department)")
_GOV_RE = re.compile(rf"(?<![a-z0-9]){_GOV} of (?:the )?({_alt(_ALIAS)})(?![a-z0-9])"
                     rf"|(?<![a-z0-9])({_alt(_ALIAS)}) {_GOV}(?![a-z0-9])")

# Uses the right COW code for the year: Germany 1955-1989 = 260 (West Germany), Yemen before 1990 = 678,
# Czech Republic or Slovakia before 1993 = 315 (Czechoslovakia).
def cow_for_year(code, year):
    if year:
        if code == 255 and 1955 <= year <= 1989: return 260
        if code == 260 and not 1955 <= year <= 1989: return 255
        if code == 679 and year < 1990: return 678
        if code in (316, 317) and year < 1993: return 315
    return code

# Country name for a COW code, e.g. 750 -> 'India'.
def cow_name(code):
    return COW.get(code, (str(code),))[0]

# Finds every country named in a piece of text (official names, common short forms, words like 'Japanese',
# and government aid agencies like USAID) and returns their COW codes in order.
def states_in(text, year=None):
    """All COW states named in a piece of text, in order: [(code, matched words)]."""
    out = []
    for m in _STATE_RE.finditer(_fold_name(text)):
        code = cow_for_year(_AGENCY.get(m.group(1)) or _ALIAS[m.group(1)], year)
        if code not in [c for c, _ in out]:
            out.append((code, m.group(1)))
    return out

# Checks whether an NGO or IO entry is really a foreign government (USAID, JICA, 'Embassy of Japan',
# 'Government of India') and returns that country's COW code; None if it is not.
def state_agency_in(text, year=None):
    """A state's own agency or government listed as an NGO/IO ('USAID', 'Embassy of Japan') -> COW code."""
    f = _fold_name(text)
    m = _AGENCY_RE.search(f)
    if m:
        return cow_for_year(_AGENCY[m.group(1)], year)
    m = _GOV_RE.search(f)
    return cow_for_year(_ALIAS[m.group(1) or m.group(2)], year) if m else None

HOME_COW = next((c for c, _ in states_in(COUNTRY)), None)
print(f"{len(COW)} COW states loaded | {COUNTRY} = {HOME_COW}")
if HOME_COW is None:
    print(f"⚠️ no COW code found for {COUNTRY!r}: add it to COW_ALIASES in the coding-rules cell "
          f"(e.g. \"{COUNTRY.lower()}\": 771)")

# ---------- reading answers ----------
# Gets the coder's answer to a question from a JSON by its short name, e.g. 'gov_info_level' finds '20_gov_info_level'.
def jget(doc, short):
    """Value of a code-sheet question in a JSON, found by its short name ('gov_info_level' -> '20_gov_info_level')."""
    for sec in doc.values():
        if isinstance(sec, dict):
            for k, v in sec.items():
                if k.split("_", 1)[-1] == short and re.match(r"\d+_", k):
                    return v
    return None

# Gets the full key of a question in a JSON, e.g. 'gov_actors' -> '24_gov_actors' (used on the Coding_Audit sheet).
def jkey(doc, short):
    for sec in doc.values():
        if isinstance(sec, dict):
            for k in sec:
                if k.split("_", 1)[-1] == short and re.match(r"\d+_", k):
                    return k
    return short

# True if an answer means 'no information': blank, '--', 'N/A', 'unknown', 'not reported' and similar (coded -9).
def is_missing_answer(v):
    """None, '', '--', 'N/A', 'unknown', 'no information (-9)' ... -> True."""
    if is_blank(v):
        return True
    s = fold(v).strip(" .")
    return any(s == w or (w[0].isalpha() and re.match(re.escape(w) + r"\b", s)) for w in MISSING_ANSWERS)

# Turns a coded answer into its number: 'Medium (2)' -> 2, 'H' -> 3, 'Yes (1)' -> 1, '--' -> -9.
# If the word and number disagree (e.g. 'No (1)') or the answer can't be read, it gives no code and a
# problem to list on the Review sheet.
def answer_code(v, kind):
    """A coded answer ('Medium (2)', 'H', 'Yes (1)', 'No', '--', None) -> (code or None, problem or None)."""
    valid = {0, 1, 2, 3} if kind == "level" else {0, 1}
    if isinstance(v, list):
        v = "; ".join(str(x) for x in v) if v else None
    if is_missing_answer(v):
        return MISSING, None
    if isinstance(v, (int, float)) and not isinstance(v, bool):
        n = int(v) if float(v).is_integer() else None
        if n == MISSING or n in valid:
            return n, None
        return None, f"{v!r} is not a valid code"
    s = fold(v)
    words = LEVEL_WORDS if kind == "level" else YESNO_WORDS
    m = re.match(r"\s*(?:(-?\d+)\b|([a-z]+))", s)
    from_word = words.get(m.group(2)) if m and m.group(2) else None
    paren = re.search(r"\((-?\d+)\)", s)
    from_num = int(paren.group(1)) if paren else (int(m.group(1)) if m and m.group(1) else None)
    if from_num is not None and from_num != MISSING and from_num not in valid:
        return None, f"{v!r}: {from_num} is not a valid code"
    if from_word is not None and from_num is not None and from_word != from_num:
        return None, f"{v!r}: the word and the number disagree"
    code = from_word if from_word is not None else from_num
    if code is None:
        return None, f"cannot read {str(v)[:80]!r} as {'None/Low/Medium/High' if kind == 'level' else 'Yes/No'}"
    return code, None

# Splits a list answer (actors, leaders, states) into separate entries, drops blanks and repeats, and says
# whether the coder listed something, wrote 'None', or left it missing.
def list_entries(v):
    """A list answer -> (separate entries, status 'listed' | 'none' | 'missing', duplicates dropped)."""
    if isinstance(v, list):
        items = [str(x).strip() for x in v if not is_blank(x)]
    elif is_blank(v):
        items = []
    else:
        items = [p.strip() for p in re.split(r"\n|;", str(v)) if p.strip()]
    real, dups, seen, saw_none = [], [], set(), False
    for it in items:
        f = fold(it).strip(" .")
        if is_missing_answer(it):
            continue
        if f in NONE_ENTRIES:
            saw_none = True
            continue
        key = re.sub(r"[^a-z0-9]+", " ", f).strip()
        if key in seen:
            dups.append(it)
            continue
        seen.add(key)
        real.append(it)
    return real, ("listed" if real else "none" if saw_none else "missing"), dups

# Checks one government actor against the military, police or civil-defence keyword list and returns the
# keyword that matched (None if none did).
def _kw_match(kind, text):
    rule = ACTOR_KEYWORDS[kind]
    low = " " + re.sub(r"[^a-z0-9]+", " ", fold(text)) + " "
    if any(" " + re.sub(r"[^a-z0-9]+", " ", w.lower()).strip() + " " in low for w in rule.get("exclude", [])):
        return None
    for w in rule["words"]:
        if " " + re.sub(r"[^a-z0-9]+", " ", w.lower()).strip() in low:
            return w
    for a in rule.get("acronyms", []):
        if re.search(rf"(?<![A-Za-z0-9]){re.escape(a)}(?![A-Za-z0-9])", str(text)):
            return a
    return None

# ---------- coding one code sheet ----------
# Codes questions 24-28: counts government actors, leaders, NGOs, IOs and other states; sets the military,
# police and civil-defence dummies; moves foreign government agencies listed as NGOs/IOs to the states and
# looks up COW codes. Every entry and how it was counted goes to the Coding_Audit sheet.
def code_actors(doc, year, disno):
    """Questions 24-28 -> {(question, kind): value}, audit rows, issues."""
    out, audit, issues = {}, [], []
    note = lambda sev, var, msg, val=None: issues.append({"severity": sev, "variable": var, "issue": msg, "value": val})
    lists = {s: list_entries(jget(doc, s)) for s in
             ("gov_actors", "officials_visited", "ngo_actors", "io_actors", "other_states")}
    # question 24: count + military / police / civil defence
    g_entries, g_status, _ = lists["gov_actors"]
    out[("gov_actors", "count")] = len(g_entries) if g_status == "listed" else (0 if g_status == "none" else MISSING)
    for e in g_entries:
        tags = [k for k in ("military", "police", "civildef") if _kw_match(k, e)]
        audit.append((disno, jkey(doc, "gov_actors"), e, "counted" + (f"; {', '.join(tags)}" if tags else "")))
        foreign = [c for c, _ in states_in(e, year) if c != HOME_COW]
        if foreign:
            note("check", "govtactors", f"question 24 lists {e!r}, which names another state "
                 f"({', '.join(cow_name(c) for c in foreign)}) — should it be under other states (28)?")
    for kind in ("military", "police", "civildef"):
        out[("gov_actors", kind)] = (MISSING if g_status == "missing"
                                     else int(any(_kw_match(kind, e) for e in g_entries)))
    # questions 25-27: counts (state agencies listed as NGOs/IOs are moved to the states)
    moved = []
    for s in ("officials_visited", "ngo_actors", "io_actors"):
        entries, status, _ = lists[s]
        keep = []
        for e in entries:
            code = state_agency_in(e, year) if s != "officials_visited" else None
            if code and code == HOME_COW:
                note("check", s, f"{e!r} is {COUNTRY}'s own government — should it be under question 24?")
            if code and code != HOME_COW and RECATEGORIZE_STATE_AGENCIES:
                moved.append((e, code, s))
                audit.append((disno, jkey(doc, s), e, f"a state agency: counted as {cow_name(code)} ({code}) "
                                                      f"under states, not here"))
                note("info", s, f"{e!r} counted as the state {cow_name(code)} ({code}), not as an NGO/IO")
                continue
            keep.append(e)
            if s != "officials_visited":
                audit.append((disno, jkey(doc, s), e, "counted"))
        if s == "officials_visited":
            audit += [(disno, jkey(doc, s), e, "counted") for e in entries]
        out[(s, "count")] = len(keep) if status == "listed" else (0 if status == "none" else MISSING)
    # question 28: other states -> count + COW codes
    s_entries, s_status, _ = lists["other_states"]
    codes, unresolved = [], []
    for e in s_entries:
        found = [c for c, _ in states_in(e, year)]
        other = [c for c in found if c != HOME_COW]
        if not found:
            unresolved.append(e)
            audit.append((disno, jkey(doc, "other_states"), e, "no COW state found: counted as 1 state, no code"))
            note("check", "states", f"cannot match {e!r} to a COW state — counted as one state without a code; "
                 f"if it is not a state (e.g. the EU), it belongs under IOs")
        elif not other:
            audit.append((disno, jkey(doc, "other_states"), e, f"{COUNTRY} itself: not counted"))
            note("check", "states", f"{e!r} is {COUNTRY} itself — not counted as another state")
        else:
            for c in other:
                audit.append((disno, jkey(doc, "other_states"), e,
                              f"COW {c} ({cow_name(c)})" + ("" if c not in codes else ": same state as above")))
                if c not in codes:
                    codes.append(c)
            if len(other) > 1:
                note("info", "states", f"{e!r} names {len(other)} states; each is counted")
    for e, c, s in moved:
        if c not in codes:
            codes.append(c)
    if s_status == "missing" and not moved:
        out[("other_states", "states")] = (MISSING, [])
    else:
        n = len(codes) + len(unresolved)
        out[("other_states", "states")] = (n, codes)
        if s_status == "listed" and n != len(s_entries):
            note("info", "states", f"{len(s_entries)} entries in question 28 counted as {n} states "
                 f"(the same state listed twice, a state agency, or {COUNTRY} itself)")
    for s, (_, _, dups) in lists.items():
        for d in dups:
            audit.append((disno, jkey(doc, s), d, "listed again: counted once"))
    return out, audit, issues

# Codes every new table variable for one disaster from its JSON (levels, yes/no, counts, text) and lists
# anything that couldn't be coded, or a country or COW code on the sheet that doesn't match the folder's country.
def code_codesheet(doc, disno, year):
    """One JSON -> (values of the new variables, audit rows, issues)."""
    actors, audit, issues = code_actors(doc, year, disno)
    vals = {}
    for col, short, kind in TABLE_VARIABLES:
        v = jget(doc, short)
        if kind == "text":
            vals[col] = "; ".join(map(str, v)) if isinstance(v, list) else v
        elif kind == "disno":
            vals[col] = disno
        elif kind == "ccode":
            vals[col] = HOME_COW
            n = to_number(v)
            if not is_missing_answer(v) and n != HOME_COW:
                issues.append({"severity": "check", "variable": col, "value": v,
                               "issue": f"COW code on the sheet is {v!r}; {COUNTRY} is {HOME_COW} (used)"})
        elif kind in ("level", "yesno"):
            code, problem = answer_code(v, kind)
            vals[col] = code
            if problem:
                issues.append({"severity": "check", "variable": col, "value": v,
                               "issue": f"{problem} — left empty; set it in Table_Overrides"})
        elif kind == "states":
            vals[col], vals["_state_codes"] = actors[(short, "states")]
        elif (short, kind) in actors:
            vals[col] = actors[(short, kind)]
        elif kind == "count":
            entries, status, _ = list_entries(v)
            vals[col] = len(entries) if status == "listed" else (0 if status == "none" else MISSING)
        else:
            raise ValueError(f"unknown kind {kind!r} for {col}")
    country = jget(doc, "country_name")
    if country and fold(country) != fold(COUNTRY):
        issues.append({"severity": "check", "variable": "cname", "value": country,
                       "issue": f"country on the sheet is {country!r}, the folder is {COUNTRY!r}"})
    return vals, audit, issues

# ---------- the three exceptions: coder's dates, deaths/affected, declaration ----------

# The three exceptions in the instructions: where the coder's dates, deaths/total affected or declaration
# differ from EM-DAT, the coder's value replaces EM-DAT's. Unreadable or implausible values are not
# used and are flagged instead.
def coder_revisions(doc, row, cols):
    """-> (changes [(column, EM-DAT value, code-sheet value, why)], issues)."""
    changes, issues = [], []
    note = lambda sev, var, msg, val=None: issues.append({"severity": sev, "variable": var, "issue": msg, "value": val})
    g = lambda f: to_number(row.get(cols[f])) if cols.get(f) else None
    if APPLY_CODER_REVISIONS["dates"]:
        text = jget(doc, "disaster_dates")
        if not is_missing_answer(text):
            p = parse_dates(text)
            em = [g(f) for f in DATE_FIELDS]
            em_year = em[0] or to_number(dis_key(row.get(cols["disno"]))[0][:4])
            if p["issue"]:
                note("check", "dates", f"code-sheet dates not used: {p['issue']}", text)
            elif em_year and abs(p["start"][0] - em_year) > MAX_YEAR_CHANGE:
                note("check", "dates", f"the sheet's start year {p['start'][0]} is far from EM-DAT's {em_year} "
                     f"(typo?) — EM-DAT dates kept", text)
            else:
                new = list(p["start"]) + (list(p["end"]) if p["end"] else [None] * 3)
                merged = [n if n is not None else e for n, e in zip(new, em)]
                k = lambda t: (t[0] or 0, t[1] or 0, t[2] or 0)
                if merged[3] and k(merged[3:]) < k(merged[:3]):
                    note("check", "dates", f"with the sheet's dates the end would be before the start "
                         f"({merged[:3]} > {merged[3:]}) — EM-DAT dates kept", text)
                else:
                    for f, n, e in zip(DATE_FIELDS, new, em):
                        if n is not None and n != e and cols.get(f):
                            changes.append((cols[f], e, n, f"code-sheet dates {text!r}"))
                    if p.get("note"):
                        note("info", "dates", p["note"], text)
    for short, field, key in (("total_deaths", "deaths", "deaths"), ("total_affected", "affected", "affected")):
        v = jget(doc, short)
        if not APPLY_CODER_REVISIONS[key] or is_missing_answer(v) or not cols.get(field):
            continue
        n = to_number(v)
        if n is None:
            note("check", cols[field], f"{short} on the sheet is not a plain number — EM-DAT value kept", v)
        elif n != g(field):
            changes.append((cols[field], row.get(cols[field]), n, f"code sheet {short}"))
    v = jget(doc, "declaration_of_disaster")
    if APPLY_CODER_REVISIONS["declaration"] and not is_missing_answer(v) and cols.get("declaration"):
        w, old = yes_no_word(v), row.get(cols["declaration"])
        if w is None:
            note("check", cols["declaration"], "declaration on the sheet does not start with Yes/No — EM-DAT kept", v)
        elif w != yes_no_word(old):
            changes.append((cols["declaration"], old, w, f"code-sheet declaration {v!r}"))
    return changes, issues

# ---------- disasters that are not on EM-DAT sheet 1 ----------
# Builds a new row for a disaster that is not on EM-DAT sheet 1 (e.g. a lab-found YYYY-0000 disaster),
# filling the EM-DAT columns from the code sheet where possible.
def row_from_codesheet(doc, disno, base_cols, base_df):
    """A new row (YYYY-0000-ISO, or a DisNo not in EM-DAT): EM-DAT columns filled from the code sheet."""
    row, filled = {c: None for c in base_cols}, []
    def put(field, value):
        col = emdat_col(base_cols, field)
        if col and not is_missing_answer(value):
            row[col] = value
            filled.append(col)
    event, iso = dis_key(disno)
    put("disno", disno); put("year", int(event[:4])); put("seq", event[5:])
    put("country", jget(doc, "country_name") or COUNTRY); put("iso", iso or HOME_ISO)
    for f in ("region", "continent", "subregion"):
        col = emdat_col(base_cols, f)
        if col and base_df[col].notna().any():
            put(f, base_df[col].mode().iloc[0])
    for short, field in (("disaster_group", "group"), ("disaster_subgroup", "subgroup"), ("disaster_type", "type"),
                         ("disaster_subtype", "subtype"), ("event_name", "event_name"),
                         ("disaster_location", "location"), ("disaster_origin", "origin"),
                         ("river_basin", "river_basin")):
        put(field, jget(doc, short))
    for short, field in (("ofda_response", "ofda"), ("appeal_intl_assistance", "appeal"),
                         ("declaration_of_disaster", "declaration")):
        put(field, yes_no_word(jget(doc, short)))
    p = parse_dates(jget(doc, "disaster_dates"))
    if not p["issue"]:
        for f, v in zip(DATE_FIELDS, list(p["start"]) + list(p["end"] or (None, None, None))):
            if v is not None:
                put(f, v)
    for short, field in (("total_deaths", "deaths"), ("total_affected", "affected")):
        put(field, to_number(jget(doc, short)))
    return row, filled

# Copies a disaster's row from another EM-DAT sheet (e.g. sheet 2), matching columns by name.
def row_from_other_sheet(rec, base_cols):
    other = {norm_col(k): v for k, v in rec["row"].items()}
    return {c: other.get(norm_col(c)) for c in base_cols}

# ---------- reading the JSON folder ----------
# Reads every code-sheet JSON in Data Files/JSONs (names starting with '_' are skipped);
# files that can't be read are listed on the Review sheet.
def load_json_docs():
    docs, issues = [], []
    for e in storage.list_entries(JSON_FOLDER):
        if e["is_dir"] or not e["name"].lower().endswith((".json", ".txt")) or e["name"].startswith(("_", "~$", ".")):
            continue
        try:
            doc = json.loads(storage.read_bytes(e["path"]).decode("utf-8-sig"))
            assert isinstance(doc, dict) and any(isinstance(v, dict) and any(re.match(r"\d+_", k) for k in v)
                                                 for v in doc.values()), "not a code-sheet JSON"
        except Exception as ex:
            if e["name"].lower().endswith(".json"):
                issues.append({"Dis No": None, "json_file": e["name"], "severity": "check", "variable": "(file)",
                               "issue": f"could not read this JSON: {type(ex).__name__}: {ex}", "value": None})
            continue
        docs.append({"name": e["name"], "path": e["path"], "doc": doc})
    return docs, issues

# The Dis No a JSON belongs to: the one in EMDAT_Record (reviewers can correct it there),
# otherwise the one typed on the code sheet.
def doc_disno(doc):
    raw = (doc.get("EMDAT_Record") or {}).get("Dis No") or jget(doc, "emdat_disno")
    event, iso = dis_key(raw)
    return disno_text(event, iso or HOME_ISO) if event else None

# Turns lists and nested values into one line of text for an Excel cell (list items joined with ' | ').
def _cell_text(v):
    if isinstance(v, list):
        return " | ".join(json.dumps(x, ensure_ascii=False) if isinstance(x, (dict, list)) else str(x) for x in v)
    if isinstance(v, dict):
        return json.dumps(v, ensure_ascii=False, default=str)
    return v

# Turns one JSON into one row of the Codesheet_Fields sheet: every answer exactly as transcribed, plus its
# review status.
def flatten_doc(doc):
    """Every field of a JSON as one row (lists joined with ' | ')."""
    out = {}
    for root, val in doc.items():
        if root in ("Extraction_Info", "ReliefWeb_Match_Info", "EMDAT_Record", "Table_Overrides"):
            continue
        if isinstance(val, dict):
            numbered = bool(val) and all(re.match(r"\d+_", k) for k in val)
            for k, v in val.items():
                out[k if numbered else f"{root}.{k}"] = _cell_text(v)
        else:
            out[root] = _cell_text(val)
    ei = doc.get("Extraction_Info") or {}
    out.update({"source_file": _name(ei.get("source_file") or ""), "extracted_at_utc": ei.get("extracted_at_utc"),
                "needs_human_review": ei.get("needs_human_review"), "human_reviewed": ei.get("human_reviewed"),
                "reviewer": ei.get("reviewer"), "review_reasons": "\n".join(ei.get("review_reasons") or [])})
    return out

# 1 if the ReliefWeb pipeline found OCHA reports for this disaster, 0 if it found none, empty if it was not run.
def _ocharep(doc):
    for block in ("OCHA_API_Report_FileName", "ReliefWeb_Match_Info"):
        n = to_number((doc.get(block) or {}).get("n_reports"))
        if n is not None:
            return int(n > 0)
    return None

# ---------- the whole table ----------
# Builds the whole table: every EM-DAT sheet-1 disaster, with each JSON attached to its row (if two JSONs
# share a Dis No, the reviewed one is used, then the newest). Applies the coder's revisions, the coding
# rules and any Table_Overrides, and collects the Review, Coding_Audit and Codesheet_Fields lists.
def build_table():
    base = EMDAT_BASE
    base_cols = list(base.columns)
    cols = {f: emdat_col(base_cols, f) for f in EMDAT_COLS}
    rows = base.to_dict("records")
    meta = [{"kind": "emdat", "changes": [], "filled": [], "doc": None} for _ in rows]
    pos = {}
    for i, r in enumerate(rows):
        ev, iso = dis_key(r.get(cols["disno"]))
        pos[disno_text(ev, iso or HOME_ISO)] = i
    docs, issues = load_json_docs()
    audit, fields = [], []
    issue = lambda disno, fname, d: issues.append({"Dis No": disno, "json_file": fname, **d})

    groups = {}
    for d in docs:
        key = doc_disno(d["doc"])
        if key is None:
            issue(None, d["name"], {"severity": "check", "variable": "emdat", "value": jget(d["doc"], "emdat_disno"),
                                    "issue": "no readable DisNo — not in the table; put it in EMDAT_Record['Dis No']"})
            fields.append({"Dis No": None, "json_file": d["name"], "used_in_table": 0, **flatten_doc(d["doc"])})
            continue
        groups.setdefault(key, []).append(d)
    chosen, dup_keys, excluded = [], {}, 0
    for key, items in groups.items():
        if key.split("-")[1] == "0000":                 # lab-found disasters: every JSON is its own disaster
            chosen += [(key, d) for d in items]
            continue
        items.sort(key=lambda d: str((d["doc"].get("Extraction_Info") or {}).get("extracted_at_utc") or ""), reverse=True)
        items.sort(key=lambda d: not (d["doc"].get("Extraction_Info") or {}).get("human_reviewed"))
        chosen.append((key, items[0]))
        if len(items) > 1:
            dup_keys[key] = [d["name"] for d in items[1:]]
        for d in items[1:]:
            issue(key, d["name"], {"severity": "check", "variable": "(file)", "value": None,
                                   "issue": f"another JSON for the same disaster ({items[0]['name']}) was used; "
                                            f"delete or rename this one if it is out of date"})
            fields.append({"Dis No": key, "json_file": d["name"], "used_in_table": 0, **flatten_doc(d["doc"])})

    for key, d in sorted(chosen, key=lambda kd: kd[0]):
        doc, fname = d["doc"], d["name"]
        rec = None if key in pos else find_emdat(key)[0]
        if ONLY_BASELINE_SHEET and rec is not None and rec["sheet"] != EMDAT_BASE_SHEET:
            fields.append({"Dis No": key, "json_file": fname, "used_in_table": 0, **flatten_doc(doc)})
            issue(key, fname, {"severity": "info", "variable": "(row)", "value": rec["sheet"],
                               "issue": f"left out of the table: {emdat_get(rec['row'], 'type')} on EM-DAT sheet "
                                        f"{rec['sheet']!r}, outside the baseline sheet (ONLY_BASELINE_SHEET = True)"})
            excluded += 1
            continue
        fields.append({"Dis No": key, "json_file": fname, "used_in_table": 1, **flatten_doc(doc)})
        before = len(issues)
        if key in dup_keys:
            issue(key, fname, {"severity": "check", "variable": "(file)", "value": None,
                               "issue": f"{len(dup_keys[key]) + 1} JSONs for this disaster; this one was used "
                                        f"(human-reviewed first, then the newest). Others: {dup_keys[key]}"})
        if key in pos and key.split("-")[1] != "0000":
            i = pos[key]
        else:
            rec, how = find_emdat(key)
            if rec is not None and rec["sheet"] != EMDAT_BASE_SHEET:
                rows.append(row_from_other_sheet(rec, base_cols))
                meta.append({"kind": "other_sheet", "changes": [], "filled": [], "doc": None})
                issue(key, fname, {"severity": "check", "variable": "(row)", "value": rec["sheet"],
                                   "issue": f"on EM-DAT sheet {rec['sheet']!r}, not the baseline sheet — added as an "
                                            f"extra row with that sheet's values"})
            else:
                row, filled = row_from_codesheet(doc, key, base_cols, base)
                rows.append(row)
                meta.append({"kind": "codesheet", "changes": [], "filled": filled, "doc": None})
                lab = key.split("-")[1] == "0000"
                issue(key, fname, {"severity": "check", "variable": "(row)", "value": None,
                                   "issue": ("not in EM-DAT (found by the lab)" if lab else f"{how}")
                                            + f" — added as a new row; EM-DAT columns filled from the code sheet "
                                              f"({len(filled)}), money and latitude/longitude left empty: please check"})
            i = len(rows) - 1
        row, m = rows[i], meta[i]
        m["doc"] = d
        year = to_number(row.get(cols["year"])) if cols.get("year") else None
        year = year or int(key[:4])
        if m["kind"] != "codesheet":                    # the three exceptions
            changes, iss = coder_revisions(doc, row, cols)
            for c, old, new, why in changes:
                row[c] = new
                m["changes"].append((c, old, new, why))
            for x in iss:
                issue(key, fname, x)
        vals, aud, iss = code_codesheet(doc, key, year)
        audit += aud
        for x in iss:
            issue(key, fname, x)
        ei = doc.get("Extraction_Info") or {}
        if ei.get("needs_human_review") and not ei.get("human_reviewed"):
            for r in ei.get("review_reasons") or []:
                issue(key, fname, {"severity": "check", "variable": "(extraction)", "value": None, "issue": r})
        overrides = {k: v for k, v in (doc.get("Table_Overrides") or {}).items() if not str(k).startswith("_")}
        for var, val in overrides.items():                         # a person's overrides win
            if var in base_cols:
                if not (row.get(var) == val or (to_number(row.get(var)) is not None and to_number(row.get(var)) == to_number(val))):
                    m["changes"].append((var, row.get(var), val, "Table_Overrides in the JSON"))
                row[var] = val
            elif not (var in vals or re.fullmatch(STATE_COLUMN.format(r"\d+"), var) or var in INFO_COLUMNS):
                issue(key, fname, {"severity": "check", "variable": var, "value": val,
                                   "issue": "Table_Overrides names a column that is not in the table — ignored"})
                continue
            issue(key, fname, {"severity": "info", "variable": var, "value": val, "issue": "set by Table_Overrides"})
        for x in issues[before:]:
            if x["severity"] == "check" and (x.get("variable") in overrides or overrides.get("needs_review") == 0):
                x["severity"] = "accepted"
                x["issue"] += (" [resolved by Table_Overrides]" if x.get("variable") in overrides
                               else " [accepted: needs_review = 0 in Table_Overrides]")
        checks = any(x["severity"] == "check" for x in issues[before:])
        vals.update({"ocharep": _ocharep(doc), "codesheet": 1, "coder": doc.get("Coder_Name"),
                     "codesheet_notes": " | ".join(map(str, doc.get("Notes") or [])) or None,
                     "emdat_changes": "; ".join(f"{c}: {'(blank)' if is_blank(o) else o} -> {n}"
                                                for c, o, n, _ in m["changes"]) or None,
                     "needs_review": int(checks), "json_file": fname})
        vals.update({k: v for k, v in overrides.items() if k not in base_cols})
        m["vals"] = vals

    # rows without a code sheet, the state columns, column order
    new_cols = [col for col, _, _ in TABLE_VARIABLES]
    n_states = max([len(m.get("vals", {}).get("_state_codes") or []) for m in meta] + [1])
    state_cols = [STATE_COLUMN.format(k + 1) for k in range(n_states)]
    sidx = next((j for j, (c, _, k) in enumerate(TABLE_VARIABLES) if k == "states"), None)
    if sidx is not None:
        new_cols = new_cols[:sidx + 1] + state_cols + new_cols[sidx + 1:]
    out_cols = base_cols + [c for c in new_cols + INFO_COLUMNS if c not in base_cols]
    for row, m in zip(rows, meta):
        vals = m.get("vals")
        if vals is None:
            vals = {c: NOT_CODED for c in new_cols}
            ev, iso = dis_key(row.get(cols["disno"]))
            vals.update({"cname": row.get(cols["country"]) if cols.get("country") else COUNTRY, "ccode": HOME_COW,
                         "emdat": disno_text(ev, iso or HOME_ISO), "codesheet": 0})
        codes = vals.pop("_state_codes", None) or []
        for k, c in enumerate(state_cols):
            vals.setdefault(c, codes[k] if k < len(codes) else None)
        for c in new_cols + INFO_COLUMNS:
            row[c] = vals.get(c)
    order = sorted(range(len(rows)), key=lambda j: (str(rows[j].get(cols["disno"])),
                                                     *[to_number(rows[j].get(cols[f])) or 0 for f in DATE_FIELDS[:3]]))
    rows, meta = [rows[j] for j in order], [meta[j] for j in order]
    return {"columns": out_cols, "n_emdat_cols": len(base_cols), "rows": rows, "meta": meta, "issues": issues,
            "audit": audit, "fields": fields, "n_json": len(docs), "disno_col": cols["disno"], "excluded": excluded}

# ---------- writing the workbook ----------
_HDR = Font(bold=True)
_FILL = {"emdat": PatternFill("solid", fgColor="D9D9D9"), "new": PatternFill("solid", fgColor="E2EFDA"),
         "info": PatternFill("solid", fgColor="FCE4D6"), "changed": PatternFill("solid", fgColor="FFFF00"),
         "filled": PatternFill("solid", fgColor="DDEBF7")}

# Makes a value safe for Excel: removes characters Excel can't store, trims very long text, turns lists into text.
def _xl(v):
    if is_blank(v):
        return None
    if isinstance(v, str):
        return ILLEGAL_CHARACTERS_RE.sub("", v)[:32000]
    if isinstance(v, (list, dict)):
        return _xl(_cell_text(v))
    if hasattr(v, "item") and not isinstance(v, (datetime.date, datetime.time)):   # numpy numbers
        return v.item()
    return v

# Adds a simple sheet to the workbook: bold header row, frozen top row, filter buttons and column widths.
def _sheet(wb, title, header, rows, widths=None, fill=None):
    ws = wb.create_sheet(title)
    ws.append(header)
    for c in ws[1]:
        c.font = _HDR
        if fill:
            c.fill = fill
    for r in rows:
        ws.append([_xl(x) for x in r])
    ws.freeze_panes = "A2"
    if rows:
        ws.auto_filter.ref = ws.dimensions
    for j, h in enumerate(header, 1):
        ws.column_dimensions[get_column_letter(j)].width = (widths or {}).get(h, min(max(len(str(h)) + 2, 10), 40))
    return ws

# Writes the Codebook: for every column, where it comes from (EM-DAT or code sheet) and how it is coded.
def codebook_rows(t):
    explain = {
        "text": "copied from the code sheet as written", "disno": "EM-DAT disaster number (= Dis No)",
        "ccode": f"COW country code of {COUNTRY} ({HOME_COW}); the code sheet's value is checked against it",
        "level": "-9 missing/no information, 0 None, 1 Low, 2 Medium, 3 High",
        "yesno": "-9 missing/no information, 0 No, 1 Yes",
        "count": "number of separate entries listed (-9 nothing listed, 0 'None')",
        "military": "1 if a government actor in Q24 is military (army, navy, air force, coast guard, marines, "
                    "national guard, ...), 0 if not, -9 if Q24 is empty",
        "police": "1 if a government actor in Q24 is police, 0 if not, -9 if Q24 is empty",
        "civildef": "1 if the civil defence / protection / national disaster-management body is among the Q24 "
                    "actors (see ACTOR_KEYWORDS), 0 if not, -9 if Q24 is empty",
        "states": "number of other states in Q28 (a state counted once; state agencies listed as NGOs/IOs added; "
                  f"{COUNTRY} itself not counted), -9 if nothing listed",
    }
    example = next((m["doc"]["doc"] for m in t["meta"] if m.get("doc")), {})
    out = [(c, "EM-DAT", "", "as in EM-DAT sheet 1" + (" — replaced by the code sheet's value where it differs "
            "(cell highlighted, original in its comment)" if c in [emdat_col([c], f) for f in
            ("sy", "sm", "sd", "ey", "em", "ed", "deaths", "affected", "declaration")] else ""))
           for c in t["columns"][:t["n_emdat_cols"]]]
    for col, short, kind in TABLE_VARIABLES:
        out.append((col, "code sheet", jkey(example, short) if example else short, explain[kind]))
        if kind == "states":
            out.append((STATE_COLUMN.format("N"), "code sheet", jkey(example, short) if example else short,
                        "COW country code of the N-th other state, in the order listed"))
    out += [("ocharep", "JSON", "OCHA blocks", "1 if the ReliefWeb pipeline found OCHA reports, 0 if none, empty if not run"),
            ("codesheet", "JSON", "", "1 if this disaster has a code-sheet JSON, 0 if not yet coded"),
            ("coder", "code sheet", "Coded by", "coder name(s)"),
            ("codesheet_notes", "code sheet", "Notes", "the Notes section, joined with |"),
            ("emdat_changes", "this script", "", "EM-DAT values replaced by the code sheet (or Table_Overrides)"),
            ("needs_review", "this script", "", "1 if the Review sheet has a 'check' item for this disaster"),
            ("json_file", "this script", "", "the JSON the row was built from")]
    return out

# Writes the Excel workbook: the DisasterLevel table (yellow cell = EM-DAT value replaced, original in a
# comment; blue cell = filled from the code sheet), then Review, Coding_Audit, Codesheet_Fields, Codebook
# and About. Saves it in Data Files as <Country>DisasterLevelofAnalysis<DDMMYYYY>.xlsx.
def save_table(t):
    wb = Workbook()
    ws = wb.active
    ws.title = "DisasterLevel"
    cols, n_em = t["columns"], t["n_emdat_cols"]
    ws.append(cols)
    for j, c in enumerate(ws[1]):
        c.font = _HDR
        c.fill = _FILL["emdat"] if j < n_em else (_FILL["info"] if cols[j] in INFO_COLUMNS else _FILL["new"])
        c.alignment = Alignment(wrap_text=False)
    colpos = {c: j + 1 for j, c in enumerate(cols)}
    for i, (row, m) in enumerate(zip(t["rows"], t["meta"]), start=2):
        ws.append([_xl(row.get(c)) for c in cols])
        for c in m["filled"]:
            ws.cell(i, colpos[c]).fill = _FILL["filled"]
        if m["kind"] != "emdat":
            ws.cell(i, 1).comment = Comment("Not on EM-DAT sheet 1: " + ("EM-DAT columns filled from the code sheet."
                                            if m["kind"] == "codesheet" else "values from another EM-DAT sheet."), "DMV")
        for c, old, new, why in m["changes"]:
            cell = ws.cell(i, colpos[c])
            cell.fill = _FILL["changed"]
            cell.comment = Comment(f"EM-DAT value: {'(blank)' if is_blank(old) else old}\nReplaced by: {why}", "DMV")
    ws.freeze_panes = "B2"
    ws.auto_filter.ref = ws.dimensions
    for j, c in enumerate(cols, 1):
        ws.column_dimensions[get_column_letter(j)].width = 50 if c == "description" else min(max(len(c) + 2, 8), 24)

    sev = {"check": 0, "info": 1, "accepted": 2}
    iss = sorted(t["issues"], key=lambda x: (sev.get(x["severity"], 2), str(x.get("Dis No")), str(x.get("json_file"))))
    _sheet(wb, "Review", ["Dis No", "json_file", "severity", "variable", "issue", "value"],
           [[x.get("Dis No"), x.get("json_file"), x["severity"], x.get("variable"), x["issue"], _cell_text(x.get("value"))]
            for x in iss], widths={"issue": 90, "value": 40, "json_file": 30})
    _sheet(wb, "Coding_Audit", ["Dis No", "question", "entry on the code sheet", "coded as"], t["audit"],
           widths={"entry on the code sheet": 60, "coded as": 50, "question": 22})
    fcols = list(dict.fromkeys(k for f in t["fields"] for k in f))
    _sheet(wb, "Codesheet_Fields", fcols, [[f.get(k) for k in fcols] for f in t["fields"]])
    _sheet(wb, "Codebook", ["variable", "source", "question / field", "coding"], codebook_rows(t),
           widths={"coding": 100, "question / field": 34, "variable": 22})
    now = datetime.datetime.now(ZoneInfo(LAB_TIMEZONE))
    used = sum(1 for m in t["meta"] if m.get("doc"))
    about = [("built", now.strftime("%Y-%m-%d %H:%M %Z")), ("country", COUNTRY), ("EM-DAT file", _name(EMDAT_PATH)),
             ("EM-DAT baseline sheet", EMDAT_BASE_SHEET), ("ONLY_BASELINE_SHEET", ONLY_BASELINE_SHEET),
             ("code sheets left out (on other EM-DAT sheets)", t["excluded"]), ("JSON folder", JSON_FOLDER),
             ("JSON files read", t["n_json"]), ("disasters with a code sheet", used),
             ("disasters in the table", len(t["rows"])), ("rules version", TABLE_RULES_VERSION),
             ("coder revisions applied", json.dumps(APPLY_CODER_REVISIONS)), ("MAX_YEAR_CHANGE", MAX_YEAR_CHANGE),
             ("RECATEGORIZE_STATE_AGENCIES", RECATEGORIZE_STATE_AGENCIES), ("missing code", MISSING),
             ("colours", "grey header = EM-DAT column, green = code-sheet variable, orange = bookkeeping; "
                         "yellow cell = EM-DAT value replaced (original in the comment); blue cell = filled from the "
                         "code sheet for a disaster not on EM-DAT sheet 1")]
    about = [(k, str(v) if isinstance(v, bool) else v) for k, v in about]
    _sheet(wb, "About", ["item", "value"], about, widths={"item": 30, "value": 110})
    name = TABLE_NAME or f"{COUNTRY.replace(' ', '')}DisasterLevelofAnalysis{now:%d%m%Y}.xlsx"
    path = storage.join(DATA_FOLDER, name)
    buf = io.BytesIO()
    wb.save(buf)
    storage.write_bytes(path, buf.getvalue())
    return path

print("table builder ready")

217 COW states loaded | Bangladesh = 771
table builder ready


### 18. Build and save the Excel table
Re-reads the EM-DAT file, builds the table and saves it in Data Files as <Country>DisasterLevelofAnalysis<DDMMYYYY>.xlsx with today's date.  
Then prints how many disasters have a code sheet, how many EM-DAT values were replaced and how many items need checking, then shows the first 15 coded rows.

In [ ]:
load_emdat()                                # re-read, in case the EM-DAT file changed since the start
table = build_table()
saved = save_table(table)

checks = [x for x in table["issues"] if x["severity"] == "check"]
coded = sum(1 for m in table["meta"] if m.get("doc"))
changed = sum(1 for m in table["meta"] if m["changes"])
print(f"\n{len(table['rows'])} disasters in the table ({coded} with a code sheet, "
      f"{sum(1 for m in table['meta'] if m['kind'] == 'codesheet')} not in EM-DAT) | "
      f"{changed} with EM-DAT values replaced by the code sheet | {len(checks)} item(s) to check"
      + (f" | {table['excluded']} code sheet(s) left out: on another EM-DAT sheet" if table["excluded"] else ""))
print("saved ->", saved)

show = [c for c in [table["disno_col"]] + [v for v, _, _ in TABLE_VARIABLES][5:] if c in table["columns"]]
pd.DataFrame([r for r, m in zip(table["rows"], table["meta"]) if m.get("doc")], columns=table["columns"])[show].head(15)

EM-DAT file: _BangladeshDisastersEMDAT2023.xlsb.xlsx
  sheet 'Sheet1': 329 disasters, 53 columns
  sheet 'Sheet2': 254 disasters, 52 columns
baseline sheet 'Sheet1': 329 disasters | 583 DisNos searchable across 2 sheet(s)

329 disasters in the table (146 with a code sheet, 0 not in EM-DAT) | 39 with EM-DAT values replaced by the code sheet | 68 item(s) to check
saved -> /DisastersProject/Bangladesh/Data Files/BangladeshDisasterLevelofAnalysis08102026.xlsx


,Dis No,govinfo,shelters,tourists,reliefgoods,govtactors,military,police,civildef,leaders,...,responsefailure,coordinationlevel,govtcoordination,govtngomeetings,tempevac,permevac,moverestrict,returnrestrict,reconstresources,reconstlevel
0,1904-0003-BGD,-9.0,-9.0,-9.0,-9.0,-9,-9,-9,-9,-9,...,-9.0,-9.0,-9.0,-9,-9.0,-9,-9,-9,-9.0,--
1,1911-0004-BGD,-9.0,-9.0,-9.0,-9.0,-9,-9,-9,-9,-9,...,-9.0,-9.0,-9.0,-9,-9.0,-9,-9,-9,-9.0,--
2,1923-0003-BGD,-9.0,-9.0,-9.0,-9.0,-9,-9,-9,-9,-9,...,-9.0,-9.0,-9.0,-9,-9.0,-9,-9,-9,-9.0,--
3,1941-0003-BGD,-9.0,-9.0,-9.0,-9.0,-9,-9,-9,-9,-9,...,-9.0,-9.0,-9.0,-9,-9.0,-9,-9,-9,-9.0,--
4,1942-0008-BGD,-9.0,-9.0,-9.0,-9.0,-9,-9,-9,-9,-9,...,-9.0,-9.0,-9.0,-9,-9.0,-9,-9,-9,-9.0,--
5,1943-9002-BGD,1.0,-9.0,-9.0,-9.0,-9,-9,-9,-9,-9,...,1.0,-9.0,-9.0,-9,-9.0,-9,-9,-9,-9.0,--
6,1947-0012-BGD,-9.0,-9.0,-9.0,-9.0,-9,-9,-9,-9,-9,...,-9.0,-9.0,-9.0,-9,-9.0,-9,-9,-9,-9.0,--
7,1958-0008-BGD,-9.0,-9.0,-9.0,-9.0,-9,-9,-9,-9,-9,...,-9.0,-9.0,-9.0,-9,-9.0,-9,-9,-9,-9.0,NaN
8,1958-0014-BGD,NaN,-9.0,-9.0,-9.0,-9,-9,-9,-9,-9,...,-9.0,-9.0,-9.0,-9,NaN,-9,-9,-9,-9.0,--
9,1960-0001-BGD,-9.0,-9.0,-9.0,-9.0,-9,-9,-9,-9,-9,...,-9.0,-9.0,-9.0,-9,-9.0,-9,-9,-9,-9.0,--
